<a id="top"></a>
# 🎓 Klasifikasi Dropout dan Keberhasilan Akademik Mahasiswa
### Skenario Prediksi Dini dan Penanganan Data Tidak Seimbang: Perbandingan Random Forest, XGBoost, dan Model Pembanding

| | |
|---|---|
| **Mata kuliah** | Data Mining E |
| **Dosen pengampu** | Ni Kadek Dwi Rusjayanthi, S.T., M.T. |
| **Program studi** | Teknologi Informasi, Fakultas Teknik, Universitas Udayana (2026) |
| **Kelompok** | Kendra Aurellia Renata (2405551118) · Tania Gracia Siwalette (2405551116) · Gerald Hizkia Turnip (2405551081) · Kezia Ardela Pinem (2405551113) |
| **Dataset** | *Predict Students' Dropout and Academic Success* — UCI ML Repository (ID 697), 4.424 mahasiswa, 36 fitur, 3 kelas |
| **Kerangka kerja** | CRISP-DM |

---

<a id="toc"></a>
## 📑 Daftar Isi

| Bab | Isi | Menjawab |
|---|---|---|
| [0](#bab0) | Persiapan lingkungan & konfigurasi | – |
| [1](#bab1) | Ringkasan proyek & peta rumusan masalah | – |
| [2](#bab2) | Tinjauan jurnal terkait & posisi penelitian | – |
| [3](#bab3) | **Pemahaman data (EDA)**: kualitas data, distribusi target, faktor risiko, uji statistik, korelasi | *Data Understanding* |
| [4](#bab4) | **Persiapan data**: feature engineering, skenario A/B/C, split, preprocessing | *Data Preparation* |
| [5](#bab5) | **Pemodelan**: baseline + screening 5 algoritma × 3 skenario × 3 strategi imbalance (5-fold CV) | RM-1, RM-3 |
| [6](#bab6) | **Penyetelan hyperparameter** (RandomizedSearchCV, Stratified 5-Fold) | RM-1 |
| [7](#bab7) | **Evaluasi pada data uji**: tabel kinerja, confusion matrix, ROC/PR, interval kepercayaan bootstrap | RM-1, **RM-2** |
| [8](#bab8) | **Pengaruh penanganan data tidak seimbang**: none vs class weight vs SMOTE | **RM-3** |
| [9](#bab9) | **Uji signifikansi statistik** antar model dan antar skenario (Friedman, Wilcoxon-Holm) | RM-1, RM-2 |
| [10](#bab10) | ⭐ **Analisis lanjutan**: early-warning (gain/lift, threshold), kalibrasi, learning curve, ablasi grup fitur, sensitivitas *temporal leakage*, perbandingan dengan literatur, audit subkelompok | nilai tambah |
| [11](#bab11) | **Interpretasi model**: Feature Importance, Permutation Importance, SHAP (global, per kelas, lokal) + triangulasi | **RM-4** |
| [12](#bab12) | ⭐ Prototipe sistem peringatan dini (*risk scoring*) | nilai tambah |
| [13](#bab13) | **Kesimpulan**, pengujian hipotesis proposal, keterbatasan, rekomendasi | semua RM |
| [14](#bab14) | Daftar pustaka | – |
| [15](#bab15) | Lampiran: reproduktibilitas (versi library, waktu eksekusi) | – |

> **Cara menjalankan.** *Run All* dari atas ke bawah. Letakkan `data.csv` (atau zip dari UCI) di folder yang sama dengan notebook; jika tidak ditemukan, notebook mencoba mengunduh lewat `ucimlrepo`, lalu meminta unggah manual (Google Colab).
> Waktu eksekusi penuh ± 20–30 menit pada CPU 2 core (Colab gratis). Untuk uji cepat ubah `FAST_MODE = True` di Bab 0 (± 3–4 menit, hasil tidak dipakai untuk laporan).
>
> Semua angka, tabel, dan kalimat temuan pada bab hasil **dihasilkan otomatis dari eksekusi** (bukan diketik manual), sehingga selalu konsisten dengan output.

<a id="bab0"></a>
## 0. Persiapan Lingkungan & Konfigurasi
[⬆ daftar isi](#toc)

Seluruh parameter eksperimen dikumpulkan di satu tempat agar mudah dikontrol dan diulang. *Random seed* dikunci (`SEED = 42`) untuk reproduktibilitas.

In [ ]:
# ---------- 0.1 Instalasi otomatis library yang mungkin belum ada ----------
import sys, subprocess, importlib

def _ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"Memasang {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for _p, _i in [("xgboost", None), ("imbalanced-learn", "imblearn"), ("shap", None)]:
    _ensure(_p, _i)

# ---------- 0.2 Import ----------
import os, re, time, glob, math, zipfile, warnings, itertools
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from scipy import stats
from scipy.stats import randint, uniform, loguniform

import sklearn, xgboost, imblearn, shap
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_validate, cross_val_score, RandomizedSearchCV, learning_curve)
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             precision_recall_fscore_support, matthews_corrcoef, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score, confusion_matrix,
                             classification_report, make_scorer, brier_score_loss, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_selection import mutual_info_classif
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from joblib import Parallel, delayed

T0 = time.time()

# ---------- 0.3 Konfigurasi eksperimen ----------
SEED        = 42        # kunci reproduktibilitas
TEST_SIZE   = 0.20      # data uji (stratified) -> tidak disentuh sampai evaluasi akhir
CV_FOLDS    = 5         # Stratified 5-Fold sesuai proposal
FAST_MODE   = True      # True = uji cepat (tuning & bootstrap dikurangi), JANGAN dipakai untuk laporan
CPU         = os.cpu_count() or 2

N_ITER = dict(DT=15, LR=8, KNN=10, RF=12, XGB=15)      # jumlah kombinasi RandomizedSearchCV per model
N_BOOT = 1000                                           # resampel bootstrap untuk interval kepercayaan
N_REPEATS_CV = 2                                        # pengulangan 5-fold untuk uji signifikansi
if FAST_MODE:
    N_ITER = dict(DT=3, LR=2, KNN=2, RF=2, XGB=2); N_BOOT = 200; N_REPEATS_CV = 1

# ---------- 0.4 Tampilan ----------
CLASSES = ["Dropout", "Enrolled", "Graduate"]
PAL = {"Dropout": "#C0392B", "Enrolled": "#E0A100", "Graduate": "#2471A3"}   # merah-kuning-biru (aman untuk buta warna merah-hijau)
SCEN_PAL = {"A": "#7F8C8D", "B": "#2E86C1", "C": "#1E8449"}
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 100, "axes.titleweight": "bold", "axes.titlesize": 12,
                     "figure.autolayout": False})
pd.set_option("display.max_columns", 60, "display.width", 200, "display.float_format", lambda v: f"{v:,.4f}")

print(f"Python {sys.version.split()[0]} | sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | "
      f"imblearn {imblearn.__version__} | shap {shap.__version__}")
print(f"CPU core = {CPU} | FAST_MODE = {FAST_MODE} | SEED = {SEED}")

<a id="bab1"></a>
## 1. Ringkasan Proyek & Peta Rumusan Masalah
[⬆ daftar isi](#toc)

**Masalah.** Dropout merugikan mahasiswa dan institusi, sementara intervensi biasanya terlambat. Pertanyaan praktisnya bukan hanya *"algoritma mana paling akurat?"* melainkan **"seberapa dini risiko dropout bisa dikenali, dan seberapa andal?"**

**Pendekatan.** Klasifikasi multikelas (*Dropout / Enrolled / Graduate*) dengan kerangka CRISP-DM, dievaluasi pada **tiga skenario waktu prediksi** (A: saat pendaftaran, B: + semester 1, C: + semester 2), **tiga strategi imbalance** (tanpa penanganan, *class weight*, SMOTE di dalam *pipeline* CV), dan **lima algoritma** + baseline mayoritas.

| RM | Pertanyaan (proposal) | Dijawab di |
|---|---|---|
| RM-1 | Algoritma mana terbaik memprediksi status akhir? | Bab 5–7, 9 |
| RM-2 | Seberapa besar peningkatan kinerja saat data semester 1 & 2 ditambahkan? | Bab 7, 9, 10 |
| RM-3 | Bagaimana class weight & SMOTE memengaruhi deteksi *Dropout*? | Bab 5, 8 |
| RM-4 | Faktor apa paling berpengaruh? | Bab 11 |

**Metrik utama:** *Macro-F1* (kelas setara; cocok untuk data tidak seimbang). Pendukung: recall/precision *Dropout*, weighted-F1, akurasi, balanced accuracy, MCC, ROC-AUC (OvR), confusion matrix.

**Hal-hal yang melampaui proposal (⭐):**
1. *Selection by CV, report on test* — pemilihan model/strategi hanya dari validasi silang; data uji dipakai sekali (tidak ada *test-set peeking*).
2. Interval kepercayaan **bootstrap** + uji **Friedman/Wilcoxon-Holm** untuk membuktikan perbedaan antar model/skenario *signifikan atau tidak*.
3. **Kurva gain/lift & tabel threshold**: jika kampus hanya sanggup memberi bimbingan ke 20% mahasiswa, berapa % dropout yang tertangkap?
4. **Kalibrasi probabilitas**, **learning curve**, **ablasi grup fitur**.
5. **Sensitivitas *temporal leakage*** pada skenario A (status pembayaran bisa baru diketahui setelah masuk).
6. **Perbandingan terukur dengan jurnal** (skenario biner agar *apple-to-apple*).
7. **Audit kewajaran subkelompok** (gender, usia, beasiswa, dll.).
8. **Triangulasi interpretasi** (Feature Importance ↔ Permutation ↔ SHAP) + prototipe *risk scoring*.

<a id="bab2"></a>
## 2. Tinjauan Jurnal Terkait & Posisi Penelitian
[⬆ daftar isi](#toc)

Sebelum membangun model, kami menelusuri publikasi yang memakai dataset yang sama atau tujuan serupa. Hanya publikasi yang **berhasil kami verifikasi sumbernya** (halaman penerbit/indeks, diakses 20 Nov 2026) yang dicantumkan. Angka yang dikutip adalah angka yang dilaporkan penulisnya dengan **protokol masing-masing**, sehingga tidak otomatis sebanding dengan hasil kami.

| # | Publikasi | Data / tugas | Metode & penanganan imbalance | Temuan utama (dilaporkan penulis) | Relevansi bagi proyek ini |
|---|---|---|---|---|---|
| 1 | Realinho, Machado, Baptista & Martins (2022). *Predicting Student Dropout and Academic Success*. **Data** (MDPI) 7(11):146 | Makalah dataset UCI-697 (IPP Portugal) | Deskripsi dataset: demografi, sosial-ekonomi, makro, akademik semester 1–2; basis *learning analytics* untuk tutor | Dataset dirancang untuk riset komparatif & pengajaran ML | **Sumber data**; dasar pembagian fitur per waktu |
| 2 | Martins, Tolledo, Machado, Baptista & Realinho (2021). *Early prediction of student's performance in higher education: a case study*. WorldCIST 2021, Springer | Dataset yang sama; prediksi dini | Model ML pada data saat masuk & akhir semester | Prediksi dini layak dilakukan | Landasan ide **skenario waktu A/B/C** |
| 3 | Villar & de Andrade (2024). *Supervised ML algorithms for predicting student dropout and academic success: a comparative study*. **Discover Artificial Intelligence** (Springer), doi:10.1007/s44163-023-00079-z | UCI-697, 3 kelas | DT, RF, SVM, GB, XGBoost, CatBoost, LightGBM; Optuna; **SMOTE** vs ADASYN; evaluasi F1 per kelas | LightGBM & CatBoost+Optuna terbaik (F1 per kelas ≈ 0,83–0,88; AUC > 0,9) | Pembanding 3-kelas + penanganan imbalance; **kami menguji kebocoran dengan SMOTE hanya di fold latih** |
| 4 | Bettahi, Belouadha & Harroud (2025). *A Modular and Explainable ML Pipeline for Student Dropout Prediction in Higher Education*. **Algorithms** 18(10):662, doi:10.3390/a18100662 | Data institusi Maroko (17.883 mhs) + replikasi pada UCI-697 (biner dropout) | 9 algoritma; *class weight*/`scale_pos_weight`; threshold per model; **SHAP** | XGBoost terbaik; replikasi UCI: F1 0,792, AUC-ROC 0,922 | Pembanding **biner** (Bab 10.6) & pendekatan *explainable* |
| 5 | Mincevičius (2026). *Interpretable ML approaches for student dropout prediction in higher education*. **New Trends in Computer Sciences** 4(1):146–160, doi:10.3846/ntcs.2026.27184 | UCI-697, **Dropout vs Graduate** (3.630 data) | LR, SVM, DT, RF, kNN, NN; SMOTE pada sebagian eksperimen; seleksi fitur SHAP | LR terbaik: ACC 0,905, F1 0,924, AUC 0,96; RF+SHAP: ACC 0,891, AUC 0,94 | Pembanding **Dropout vs Graduate**; bukti bahwa model sederhana kompetitif |
| 6 | Mustofa, Emon, Mamun, Akhy & Ahad (2025). *A novel AI-driven model for student dropout risk analysis with explainable AI insights*. **Computers and Education: AI** 8:100352, doi:10.1016/j.caeai.2024.100352 | Prediksi risiko dropout (kami tidak memverifikasi bahwa datanya identik dengan UCI-697) | *Upsampling*, RFE-CV, hibrida LR+NN (HLRNN), **SHAP & LIME** | Akurasi 96% (melampaui LR dan ANN induknya) | Rujukan metode XAI & penanganan imbalance |

**Pola yang kami tangkap dari tinjauan ini**

1. Hampir semua studi membandingkan algoritma pada **satu set fitur lengkap** (semester 1+2) — jarang menjawab *seberapa dini* prediksi bisa dipercaya.
2. Evaluasi sering berfokus pada akurasi/F1 per kelas; **kelas *Enrolled*** (kelas ambigu "di antara") jarang dibahas serius.
3. Ringkasan yang kami baca tidak selalu menjelaskan **kapan resampling diterapkan relatif terhadap pemisahan data**; padahal SMOTE sebelum *split*/CV dapat menggelembungkan skor.
4. Hasil dilaporkan sebagai satu angka tanpa **interval kepercayaan** atau **uji signifikansi**.
5. Interpretasi (SHAP) umumnya hanya satu metode.

**Posisi penelitian kami (celah yang diisi)**

| Celah di literatur (dari yang kami tinjau) | Kontribusi notebook ini |
|---|---|
| Prediksi dini tidak diukur | 3 skenario waktu + gain/lift + sensitivitas *temporal leakage* |
| Risiko kebocoran data pada resampling | SMOTE **hanya di dalam fold latih** (`imblearn.Pipeline`); preprocessing di-*fit* per fold |
| Satu angka tanpa ketidakpastian | Bootstrap CI + Friedman/Wilcoxon-Holm |
| *Enrolled* diabaikan | Analisis confusion matrix & kesalahan antar kelas |
| Satu metode interpretasi | FI + Permutation + SHAP, diuji konkordansinya |
| Hanya akurasi model | Kalibrasi, ablasi grup fitur, audit subkelompok, prototipe *risk scoring* |

> Catatan kejujuran ilmiah: tinjauan ini **bukan** *systematic review*; ia adalah penelusuran terarah untuk memposisikan proyek. Perbandingan numerik dengan literatur (Bab 10.6) dilakukan hati-hati dan hanya untuk tugas yang setara.

<a id="bab3"></a>
## 3. Pemahaman Data (*Data Understanding* / EDA)
[⬆ daftar isi](#toc)

### 3.1 Memuat data
Loader mencoba beberapa sumber secara berurutan: berkas lokal → zip UCI → `ucimlrepo` → unggah manual (Colab).

In [ ]:
def _read_csv_auto(path):
    for sep in (";", ","):
        d = pd.read_csv(path, sep=sep)
        if d.shape[1] > 5:
            return d
    raise ValueError("Format CSV tidak dikenali")

def load_data():
    # 1) berkas lokal
    for p in ["data.csv", "/content/data.csv", "/content/drive/MyDrive/data.csv", "./data/data.csv", "/home/claude/data_raw/data.csv"]:
        if os.path.exists(p):
            return _read_csv_auto(p), f"berkas lokal: {p}"
    # 2) zip dari UCI
    for z in glob.glob("*.zip") + glob.glob("/content/*.zip") + glob.glob("/home/claude/*.zip"):
        try:
            with zipfile.ZipFile(z) as zf:
                if "data.csv" in zf.namelist():
                    zf.extract("data.csv", "./data_uci")
                    return _read_csv_auto("./data_uci/data.csv"), f"zip: {z}"
        except Exception:
            pass
    # 3) ucimlrepo
    try:
        _ensure("ucimlrepo")
        from ucimlrepo import fetch_ucirepo
        r = fetch_ucirepo(id=697)
        d = r.data.features.copy()
        d["Target"] = r.data.targets.iloc[:, 0].values
        return d, "ucimlrepo (UCI ID 697)"
    except Exception as e:
        print("ucimlrepo gagal:", e)
    # 4) unggah manual di Colab
    try:
        from google.colab import files
        up = files.upload()
        return _read_csv_auto(list(up.keys())[0]), "unggah manual"
    except Exception:
        raise FileNotFoundError("data.csv tidak ditemukan. Letakkan data.csv di folder yang sama dengan notebook.")

df_raw, SRC = load_data()
df = df_raw.copy()
df.columns = [c.strip().replace("\t", "") for c in df.columns]      # ada karakter tab pada 'Daytime/evening attendance'
df = df.rename(columns={"Nacionality": "Nationality"})              # perbaikan ejaan dari sumber
print(f"Sumber data : {SRC}")
print(f"Dimensi     : {df.shape[0]:,} baris x {df.shape[1]} kolom (36 fitur + Target)")
display(df.head(3).T.head(40))

### 3.2 Kamus variabel & kelompok atribut
Kelompok atribut dipetakan sesuai proposal. **Pembagian per waktu observasi** inilah dasar skenario A/B/C.

In [ ]:
GROUPS = {
    "Demografi":        ["Marital status", "Gender", "Age at enrollment", "Nationality", "International"],
    "Jalur pendaftaran":["Application mode", "Application order", "Course", "Daytime/evening attendance",
                         "Previous qualification", "Previous qualification (grade)", "Admission grade"],
    "Latar keluarga":   ["Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"],
    "Sosial-ekonomi":   ["Scholarship holder", "Debtor", "Tuition fees up to date", "Displaced", "Educational special needs"],
    "Makroekonomi":     ["Unemployment rate", "Inflation rate", "GDP"],
    "Akademik sem-1":   [c for c in df.columns if "1st sem" in c],
    "Akademik sem-2":   [c for c in df.columns if "2nd sem" in c],
}
assert sum(len(v) for v in GROUPS.values()) == 36 and set(itertools.chain(*GROUPS.values())) == set(df.columns) - {"Target"}

# Kategorikal NOMINAL (kode angka tanpa urutan makna) -> one-hot. Sisanya numerik/biner/ordinal.
NOMINAL = ["Marital status", "Application mode", "Course", "Previous qualification", "Nationality",
           "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]
BINARY  = ["Daytime/evening attendance", "Displaced", "Educational special needs", "Debtor",
           "Tuition fees up to date", "Gender", "Scholarship holder", "International"]

COURSE_NAME = {33: "Biofuel Production Tech.", 171: "Animation & Multimedia", 8014: "Social Service (evening)",
               9003: "Agronomy", 9070: "Communication Design", 9085: "Veterinary Nursing", 9119: "Informatics Eng.",
               9130: "Equinculture", 9147: "Management", 9238: "Social Service", 9254: "Tourism", 9500: "Nursing",
               9556: "Oral Hygiene", 9670: "Advertising & Marketing", 9773: "Journalism & Communication",
               9853: "Basic Education", 9991: "Management (evening)"}

FEAT2GROUP_BASE = {c: g for g, cols in GROUPS.items() for c in cols}
summary = pd.DataFrame({"kelompok": {c: g for g, cols in GROUPS.items() for c in cols}})
summary["tipe"]   = [("nominal (one-hot)" if c in NOMINAL else "biner" if c in BINARY else "numerik/ordinal") for c in summary.index]
summary["jumlah nilai unik"] = df[summary.index].nunique()
summary["min"] = df[summary.index].min(); summary["maks"] = df[summary.index].max()
display(summary)
print({g: len(v) for g, v in GROUPS.items()})

### 3.3 Kualitas data
Dokumentasi UCI menyatakan tidak ada nilai hilang; proposal berjanji **memverifikasinya**. Kami juga memeriksa duplikat, rentang nilai, dan konsistensi logis.

In [ ]:
qc = {}
qc["Nilai hilang (total sel)"] = int(df.isna().sum().sum())
qc["Baris duplikat"]           = int(df.duplicated().sum())
qc["Admission grade di luar [0,200]"]          = int(((df["Admission grade"] < 0) | (df["Admission grade"] > 200)).sum())
qc["Previous qual. grade di luar [0,200]"]     = int(((df["Previous qualification (grade)"] < 0) | (df["Previous qualification (grade)"] > 200)).sum())
qc["Nilai sem-1/sem-2 di luar [0,20]"]         = int(((df["Curricular units 1st sem (grade)"].between(0, 20) == False) | (df["Curricular units 2nd sem (grade)"].between(0, 20) == False)).sum())
qc["Usia < 16 atau > 80"]                      = int(((df["Age at enrollment"] < 16) | (df["Age at enrollment"] > 80)).sum())
qc["Approved > Enrolled (sem-1)"]              = int((df["Curricular units 1st sem (approved)"] > df["Curricular units 1st sem (enrolled)"]).sum())
qc["Approved > Enrolled (sem-2)"]              = int((df["Curricular units 2nd sem (approved)"] > df["Curricular units 2nd sem (enrolled)"]).sum())
qc["Enrolled = 0 di sem-1"]                    = int((df["Curricular units 1st sem (enrolled)"] == 0).sum())
qc["Enrolled = 0 di sem-1 DAN sem-2"]          = int(((df["Curricular units 1st sem (enrolled)"] == 0) & (df["Curricular units 2nd sem (enrolled)"] == 0)).sum())
qc["Enrolled > 0 tetapi nilai rata-rata = 0 (sem-1)"] = int(((df["Curricular units 1st sem (enrolled)"] > 0) & (df["Curricular units 1st sem (grade)"] == 0)).sum())
display(pd.Series(qc, name="jumlah").to_frame())

# Kasus 'tidak pernah terdaftar mata kuliah' dan hubungannya dengan target
zero_both = (df["Curricular units 1st sem (enrolled)"] == 0) & (df["Curricular units 2nd sem (enrolled)"] == 0)
print("\nDistribusi target pada mahasiswa yang tidak mengambil unit di sem-1 maupun sem-2:")
display(df.loc[zero_both, "Target"].value_counts().to_frame("jumlah"))

# Outlier (IQR) - hanya dilaporkan, TIDAK dihapus (usia dewasa & nilai ekstrem adalah data sah; model pohon tahan outlier)
num_chk = ["Age at enrollment", "Admission grade", "Previous qualification (grade)", "Unemployment rate", "Inflation rate", "GDP"]
rows = []
for c in num_chk:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    rows.append((c, int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()), df[c].min(), df[c].max()))
display(pd.DataFrame(rows, columns=["fitur", "jumlah outlier IQR", "min", "maks"]).set_index("fitur"))

**Keputusan kualitas data.** Data bersih dari nilai hilang & duplikat. Temuan "unit disetujui > unit diambil" tidak dihapus karena dapat berasal dari pengakuan/kredit unit (*credited*) dan bukan kesalahan input yang pasti; nilai ekstrem usia dipertahankan karena mahasiswa dewasa adalah populasi nyata di dataset ini. Mahasiswa yang **tidak mengambil unit sama sekali** hampir semuanya *Dropout* (lihat tabel) — ini pola yang sangat prediktif dan akan terlihat lagi pada SHAP.

### 3.4 Distribusi variabel target & tingkat ketidakseimbangan

In [ ]:
tc = df["Target"].value_counts().reindex(CLASSES)
tp = (tc / tc.sum() * 100)
tdist = pd.DataFrame({"jumlah": tc, "persen": tp.round(2)})
display(tdist)
imb_ratio = tc.max() / tc.min()
print(f"Rasio ketidakseimbangan (mayoritas : minoritas) = {tc.max()} : {tc.min()} = {imb_ratio:.2f} : 1")
print(f"Akurasi 'tebak kelas mayoritas' (baseline) = {tc.max()/tc.sum():.4f}  -> akurasi saja menyesatkan")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(CLASSES, tc.values, color=[PAL[c] for c in CLASSES])
for i, v in enumerate(tc.values):
    ax[0].text(i, v + 25, f"{v:,}\n({tp.values[i]:.1f}%)", ha="center", fontsize=10)
ax[0].set_ylim(0, tc.max() * 1.18); ax[0].set_title("Distribusi kelas target"); ax[0].set_ylabel("jumlah mahasiswa")
ax[1].pie(tc.values, labels=CLASSES, colors=[PAL[c] for c in CLASSES], autopct="%1.1f%%", startangle=90,
          wedgeprops=dict(edgecolor="white", linewidth=2))
ax[1].set_title("Proporsi kelas")
plt.tight_layout(); plt.show()

### 3.5 Eksplorasi faktor risiko
Visual berikut menampilkan proporsi status akhir pada tiap kategori faktor. Batang diurutkan menurut tingkat *Dropout*.

In [ ]:
def stacked_prop(ax, series, title, order=None, labels=None, annotate_n=True):
    ct = pd.crosstab(series, df["Target"])[CLASSES]
    n = ct.sum(axis=1); ct = ct.div(n, axis=0) * 100
    ct = ct.sort_values("Dropout", ascending=True) if order is None else ct.loc[order]
    left = np.zeros(len(ct))
    for c in CLASSES:
        ax.barh(range(len(ct)), ct[c], left=left, color=PAL[c], label=c, edgecolor="white")
        left += ct[c].values
    ax.set_yticks(range(len(ct)))
    ylabs = [labels.get(i, i) if labels else i for i in ct.index]
    if annotate_n: ylabs = [f"{l}  (n={n.loc[i]})" for l, i in zip(ylabs, ct.index)]
    ax.set_yticklabels(ylabs, fontsize=9); ax.set_xlim(0, 100); ax.set_xlabel("% mahasiswa"); ax.set_title(title)

# --- per program studi ---
fig, ax = plt.subplots(figsize=(10, 6.2))
stacked_prop(ax, df["Course"], "Status akhir per program studi", labels=COURSE_NAME)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1)); plt.tight_layout(); plt.show()

# --- faktor sosial-ekonomi & demografi ---
age_grp = pd.cut(df["Age at enrollment"], [0, 19, 21, 25, 35, 100], labels=["<=19", "20-21", "22-25", "26-35", ">35"])
fig, axs = plt.subplots(2, 3, figsize=(16, 7.5))
stacked_prop(axs[0, 0], df["Tuition fees up to date"], "Uang kuliah lunas?", labels={0: "Tidak", 1: "Ya"}, order=[1, 0])
stacked_prop(axs[0, 1], df["Scholarship holder"], "Penerima beasiswa?", labels={0: "Tidak", 1: "Ya"}, order=[1, 0])
stacked_prop(axs[0, 2], df["Debtor"], "Berstatus debitur?", labels={0: "Tidak", 1: "Ya"}, order=[0, 1])
stacked_prop(axs[1, 0], df["Gender"], "Gender", labels={0: "Perempuan", 1: "Laki-laki"}, order=[0, 1])
stacked_prop(axs[1, 1], df["Displaced"], "Mahasiswa perantau (displaced)?", labels={0: "Tidak", 1: "Ya"}, order=[0, 1])
stacked_prop(axs[1, 2], age_grp, "Kelompok usia saat masuk", order=["<=19", "20-21", "22-25", "26-35", ">35"])
h, l = axs[0, 0].get_legend_handles_labels(); fig.legend(h, l, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.04, 1, 1]); plt.show()

In [ ]:
# --- kinerja akademik semester 1 & 2 menurut status akhir ---
fig, axs = plt.subplots(2, 3, figsize=(16, 7.5))
pairs = [("Curricular units 1st sem (approved)", "Unit LULUS sem-1"), ("Curricular units 2nd sem (approved)", "Unit LULUS sem-2"),
         ("Curricular units 1st sem (grade)", "Nilai rata-rata sem-1"), ("Curricular units 2nd sem (grade)", "Nilai rata-rata sem-2"),
         ("Admission grade", "Nilai masuk (admission grade)"), ("Age at enrollment", "Usia saat masuk")]
for ax, (c, t) in zip(axs.ravel(), pairs):
    sns.boxplot(data=df, x="Target", y=c, order=CLASSES, palette=PAL, ax=ax, fliersize=2, linewidth=1)
    ax.set_title(t); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

# --- rasio kelulusan unit per semester menurut status akhir ---
tmp = df.copy()
tmp["rasio lulus sem-1"] = np.where(tmp["Curricular units 1st sem (enrolled)"] > 0, tmp["Curricular units 1st sem (approved)"] / tmp["Curricular units 1st sem (enrolled)"].replace(0, np.nan), 0)
tmp["rasio lulus sem-2"] = np.where(tmp["Curricular units 2nd sem (enrolled)"] > 0, tmp["Curricular units 2nd sem (approved)"] / tmp["Curricular units 2nd sem (enrolled)"].replace(0, np.nan), 0)
display(tmp.groupby("Target")[["rasio lulus sem-1", "rasio lulus sem-2", "Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)"]].agg(["mean", "median"]).reindex(CLASSES))

# tingkat dropout menurut jumlah unit lulus semester 1 (dibinning)
b = pd.cut(df["Curricular units 1st sem (approved)"], [-1, 0, 2, 4, 6, 8, 100], labels=["0", "1-2", "3-4", "5-6", "7-8", ">8"])
dr = df.groupby(b)["Target"].apply(lambda s: (s == "Dropout").mean() * 100)
cnt = b.value_counts().reindex(dr.index)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(dr.index.astype(str), dr.values, color=PAL["Dropout"])
for i, (v, n) in enumerate(zip(dr.values, cnt.values)): ax.text(i, v + 1, f"{v:.0f}%\n(n={n})", ha="center", fontsize=9)
ax.set_ylim(0, dr.max() * 1.25); ax.set_xlabel("Jumlah unit LULUS pada semester 1"); ax.set_ylabel("% Dropout")
ax.set_title("Tingkat Dropout menurun tajam seiring unit lulus semester 1"); plt.tight_layout(); plt.show()

### 3.6 Uji asosiasi statistik terhadap target
Visual saja tidak cukup. Untuk setiap fitur dihitung **ukuran efek** terhadap `Target`:

* fitur kategorikal/biner/ordinal diskret → uji **χ²** + **Cramér's V**;
* fitur numerik kontinu → uji **Kruskal-Wallis** + **ε²** (ukuran efek non-parametrik; tidak mengasumsikan normalitas);
* ditambah **Mutual Information** (menangkap hubungan non-linear).

p-value dikoreksi **Benjamini-Hochberg (FDR)** karena 36 uji berjalan bersamaan. Ukuran efek lebih informatif daripada p-value pada n = 4.424 (hampir semua hubungan "signifikan").

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y); chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum(); k = min(ct.shape) - 1
    return chi2, p, np.sqrt(chi2 / (n * k)) if k > 0 else np.nan

def epsilon_sq(groups):
    H, p = stats.kruskal(*groups); n = sum(len(g) for g in groups); k = len(groups)
    return H, p, (H - k + 1) / (n - k)

feat_cols = [c for c in df.columns if c != "Target"]
CAT_LIKE = set(NOMINAL) | set(BINARY) | {"Application order"}
rows = []
for c in feat_cols:
    if c in CAT_LIKE:
        s, p, es = cramers_v(df[c], df["Target"]); rows.append((c, "χ² / Cramér's V", s, p, es))
    else:
        s, p, es = epsilon_sq([df.loc[df["Target"] == k, c].values for k in CLASSES]); rows.append((c, "Kruskal-Wallis / ε²", s, p, es))
assoc = pd.DataFrame(rows, columns=["fitur", "uji", "statistik", "p_value", "ukuran_efek"]).set_index("fitur")
# koreksi Benjamini-Hochberg
pv = assoc["p_value"].values; order = np.argsort(pv); m = len(pv)
adj = np.empty(m); prev = 1.0
for rank, idx in zip(range(m, 0, -1), order[::-1]):
    prev = min(prev, pv[idx] * m / rank); adj[idx] = prev
assoc["p_adj_BH"] = adj
assoc["kelompok"] = [next(g for g, cols in GROUPS.items() if c in cols) for c in assoc.index]

# Mutual information (fitur nominal sebagai kode diskret)
Xmi = df[feat_cols].copy(); ymi = LabelEncoder().fit_transform(df["Target"])
disc = [c in CAT_LIKE for c in feat_cols]
assoc["mutual_info"] = pd.Series(mutual_info_classif(Xmi, ymi, discrete_features=disc, random_state=SEED), index=feat_cols)
assoc = assoc.sort_values("ukuran_efek", ascending=False)
display(assoc.head(20)[["kelompok", "uji", "ukuran_efek", "mutual_info", "p_adj_BH"]])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
top = assoc.head(15).iloc[::-1]
ax[0].barh(top.index, top["ukuran_efek"], color=[SCEN_PAL["A"] if g not in ("Akademik sem-1", "Akademik sem-2") else SCEN_PAL["C"] for g in top["kelompok"]])
ax[0].set_title("15 fitur dengan ukuran efek terbesar\n(abu-abu = data saat pendaftaran, hijau = data akademik)"); ax[0].set_xlabel("Cramér's V / ε²")
topm = assoc.sort_values("mutual_info", ascending=False).head(15).iloc[::-1]
ax[1].barh(topm.index, topm["mutual_info"], color="#5D6D7E"); ax[1].set_title("15 fitur dengan Mutual Information terbesar"); ax[1].set_xlabel("MI (nats)")
plt.tight_layout(); plt.show()
n_sig = int((assoc["p_adj_BH"] < 0.05).sum())
print(f"{n_sig} dari {len(assoc)} fitur berasosiasi signifikan dengan target (FDR 5%).")

### 3.7 Korelasi & multikolinearitas
Korelasi tinggi antar fitur tidak merusak model pohon, tetapi memengaruhi **Logistic Regression** (koefisien tidak stabil) dan **interpretasi** importance (kontribusi terbagi antar fitur kembar). Kami memakai korelasi Spearman.

In [ ]:
num_cols = [c for c in feat_cols if c not in NOMINAL and c not in BINARY]
corr = df[num_cols].corr(method="spearman")
short = {c: c.replace("Curricular units ", "CU ").replace(" sem", "").replace("Previous qualification", "Prev.qual") for c in num_cols}
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr.rename(index=short, columns=short), cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": .7, "label": "Spearman ρ"}, linewidths=.3, ax=ax)
ax.set_title("Korelasi Spearman antar fitur numerik"); plt.tight_layout(); plt.show()

pairs_hi = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().rename("rho").reset_index()
            .query("abs(rho) >= 0.80").sort_values("rho", key=np.abs, ascending=False))
pairs_hi.columns = ["fitur 1", "fitur 2", "rho"]
display(pairs_hi.reset_index(drop=True))
print(f"{len(pairs_hi)} pasangan fitur dengan |ρ| ≥ 0,80 -> multikolinearitas kuat antar fitur akademik sem-1 dan sem-2.")

In [ ]:
top_assoc = assoc.head(5).index.tolist()
n_acad_top = sum(FEAT2GROUP_BASE[f] in ("Akademik sem-1", "Akademik sem-2") for f in top_assoc)
display(Markdown(f"""
**Ringkasan temuan EDA (otomatis)**

1. **Ketidakseimbangan:** {tdist.loc['Graduate','persen']:.1f}% Graduate, {tdist.loc['Dropout','persen']:.1f}% Dropout, {tdist.loc['Enrolled','persen']:.1f}% Enrolled (rasio {imb_ratio:.2f}:1). Akurasi baseline mayoritas = {tc.max()/tc.sum():.3f}.
2. **Kualitas data:** nilai hilang = {qc['Nilai hilang (total sel)']}, duplikat = {qc['Baris duplikat']}.
3. **Fitur dengan efek terbesar terhadap status akhir:** {', '.join('*'+f+'*' for f in top_assoc)} — {n_acad_top} dari 5 teratas adalah **data akademik semester 1–2**; ini menjadi dugaan awal bahwa skenario B/C lebih informatif daripada A (diuji pada Bab 7 dan 9).
4. **Faktor non-akademik** yang menonjol dari visual: status uang kuliah, beasiswa, usia saat masuk, dan program studi (variasi tingkat Dropout antar prodi besar).
5. **Multikolinearitas:** {len(pairs_hi)} pasangan fitur dengan |ρ| ≥ 0,8 (terutama kembar sem-1/sem-2) -> perhatikan interpretasi importance.
"""))

<a id="bab4"></a>
## 4. Persiapan Data (*Data Preparation*)
[⬆ daftar isi](#toc)

### 4.1 Rekayasa fitur (*feature engineering*)
Mengikuti proposal, ditambahkan fitur turunan yang memuat **"efisiensi studi"** dan **tren**:

| Fitur baru | Rumus | Skenario | Alasan |
|---|---|---|---|
| `grade_gap` | Admission grade − Previous qualification (grade) | A | Selisih kualitas masuk vs nilai sekolah asal |
| `sem1_approval_rate` | unit lulus ÷ unit diambil (sem-1) | B | Efisiensi studi; 0 jika tidak mengambil unit |
| `sem1_eval_pass_rate` | unit lulus ÷ jumlah evaluasi (sem-1) | B | Seberapa sering ujian berujung lulus |
| `sem2_approval_rate`, `sem2_eval_pass_rate` | idem untuk sem-2 | C | – |
| `delta_grade` | nilai sem-2 − nilai sem-1 | C | Tren naik/turun |
| `delta_approved` | unit lulus sem-2 − sem-1 | C | Tren efisiensi |
| `total_approved`, `total_approval_rate` | akumulasi sem-1+sem-2 | C | Beban kumulatif |

Pembagian nol ditangani eksplisit (hasil 0), bukan dibiarkan `NaN`/`inf`. Fitur turunan hanya memakai informasi **sebaik waktu skenarionya** sehingga tidak menimbulkan kebocoran temporal.

In [ ]:
def safe_div(a, b):
    return np.where(b > 0, a / np.where(b == 0, 1, b), 0.0)

X_all = df.drop(columns="Target").copy()
# --- turunan skenario A
X_all["grade_gap"] = X_all["Admission grade"] - X_all["Previous qualification (grade)"]
# --- turunan skenario B (sem-1)
X_all["sem1_approval_rate"]  = safe_div(X_all["Curricular units 1st sem (approved)"], X_all["Curricular units 1st sem (enrolled)"])
X_all["sem1_eval_pass_rate"] = safe_div(X_all["Curricular units 1st sem (approved)"], X_all["Curricular units 1st sem (evaluations)"])
# --- turunan skenario C (sem-2 dan gabungan)
X_all["sem2_approval_rate"]  = safe_div(X_all["Curricular units 2nd sem (approved)"], X_all["Curricular units 2nd sem (enrolled)"])
X_all["sem2_eval_pass_rate"] = safe_div(X_all["Curricular units 2nd sem (approved)"], X_all["Curricular units 2nd sem (evaluations)"])
X_all["delta_grade"]         = X_all["Curricular units 2nd sem (grade)"] - X_all["Curricular units 1st sem (grade)"]
X_all["delta_approved"]      = X_all["Curricular units 2nd sem (approved)"] - X_all["Curricular units 1st sem (approved)"]
X_all["total_approved"]      = X_all["Curricular units 1st sem (approved)"] + X_all["Curricular units 2nd sem (approved)"]
X_all["total_approval_rate"] = safe_div(X_all["total_approved"], X_all["Curricular units 1st sem (enrolled)"] + X_all["Curricular units 2nd sem (enrolled)"])

ENG_A = ["grade_gap"]
ENG_B = ["sem1_approval_rate", "sem1_eval_pass_rate"]
ENG_C = ["sem2_approval_rate", "sem2_eval_pass_rate", "delta_grade", "delta_approved", "total_approved", "total_approval_rate"]
assert np.isfinite(X_all[ENG_A + ENG_B + ENG_C].values).all()

# --- Definisi skenario waktu prediksi (pembeda utama penelitian)
ENROLL_GROUPS = ["Demografi", "Jalur pendaftaran", "Latar keluarga", "Sosial-ekonomi", "Makroekonomi"]
FEATS_A = [c for g in ENROLL_GROUPS for c in GROUPS[g]] + ENG_A
FEATS_B = FEATS_A + GROUPS["Akademik sem-1"] + ENG_B
FEATS_C = FEATS_B + GROUPS["Akademik sem-2"] + ENG_C
SCENARIOS = {"A": FEATS_A, "B": FEATS_B, "C": FEATS_C}
SCEN_DESC = {"A": "A: saat pendaftaran", "B": "B: + semester 1", "C": "C: + semester 1 & 2"}

display(pd.DataFrame({"skenario": [SCEN_DESC[k] for k in SCENARIOS], "jumlah fitur": [len(v) for v in SCENARIOS.values()],
                      "makna": ["Prediksi SEBELUM kuliah dimulai", "Prediksi SETELAH semester 1", "Prediksi SETELAH semester 2"]}).set_index("skenario"))

# Kelompok fitur lengkap (termasuk hasil rekayasa) untuk ablasi & agregasi SHAP
FULL_GROUPS = {**GROUPS, "Rekayasa fitur": ENG_A + ENG_B + ENG_C}
FEAT2GROUP = {c: g for g, cols in FULL_GROUPS.items() for c in cols}
display(X_all[ENG_A + ENG_B + ENG_C].describe().T[["mean", "std", "min", "50%", "max"]])

### 4.2 Pemisahan data (*split*)
**Stratified 80% latih / 20% uji.** Data uji **dikunci** dan baru dipakai pada Bab 7. Seluruh pemilihan model, strategi imbalance, dan hyperparameter hanya memakai **data latih** dengan Stratified 5-Fold CV.

In [ ]:
le = LabelEncoder().fit(CLASSES)                       # Dropout=0, Enrolled=1, Graduate=2
y_all = le.transform(df["Target"])
assert list(le.classes_) == CLASSES

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=SEED)
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

chk = pd.DataFrame({"seluruh": pd.Series(y_all).value_counts(normalize=True).sort_index(),
                    "latih": pd.Series(y_train).value_counts(normalize=True).sort_index(),
                    "uji": pd.Series(y_test).value_counts(normalize=True).sort_index()})
chk.index = CLASSES; chk = (chk * 100).round(2)
print(f"Data latih: {X_train.shape[0]:,} | Data uji: {X_test.shape[0]:,}  (proporsi kelas terjaga -> stratified)")
display(chk)

### 4.3 Encoding & penskalaan (di dalam *pipeline*)

* **Nominal** (`Course`, `Application mode`, kualifikasi/pekerjaan orang tua, dst.) → **One-Hot**; kategori langka (< 15 sampel) digabung sebagai *infrequent* dan kategori tak dikenal ditangani aman. Kode angka **tidak** diperlakukan sebagai urutan.
* **Biner/ordinal/numerik** → distandardisasi (penting untuk Logistic Regression & KNN; tidak merugikan pohon).
* Semua transformasi berada di **`Pipeline`** sehingga hanya di-*fit* pada fold latih (**tidak ada kebocoran statistik** dari fold validasi/uji).
* **SMOTE** (bila dipakai) disisipkan **setelah** preprocessing dan hanya aktif saat `fit` (data latih fold), tidak pernah menyentuh data validasi/uji.

> Keterbatasan yang diakui: SMOTE standar menginterpolasi di ruang fitur hasil one-hot sehingga sampel sintetis pada kolom biner/one-hot bernilai pecahan. Alternatif (SMOTE-NC) kami bahas di Bab 13 sebagai pengembangan.

<a id="bab5"></a>
## 5. Pemodelan (*Modeling*)
[⬆ daftar isi](#toc)

### 5.1 Fungsi bantu: model, pipeline, metrik
Semua eksperimen memakai **satu fabrik pipeline** yang sama agar perbandingan adil:

`ColumnTransformer (OneHot + Scaler)  →  [SMOTE opsional]  →  Model`

* *Class weight*: `class_weight="balanced"` untuk DT/LR/RF; XGBoost memakai `sample_weight` seimbang (XGBoost multikelas tidak punya `class_weight`); **KNN tidak mendukung bobot kelas** sehingga hanya diuji dengan *none* dan SMOTE.
* Rumus bobot: $w_i = \dfrac{N}{K \cdot n_i}$ (sesuai Bab II laporan).

In [ ]:
MODEL_NAMES = ["Decision Tree", "Logistic Regression", "KNN", "Random Forest", "XGBoost"]
STRATEGIES  = ["none", "class_weight", "smote"]
STRAT_LABEL = {"none": "Tanpa penanganan", "class_weight": "Class weight", "smote": "SMOTE"}

def make_estimator(name, strategy="none", params=None):
    """Estimator dengan hyperparameter default yang wajar; `params` menimpa default (prefix 'model__' dibuang)."""
    p = {k.replace("model__", ""): v for k, v in (params or {}).items()}
    cw = "balanced" if strategy == "class_weight" else None
    if name == "Decision Tree":
        base = dict(max_depth=6, min_samples_leaf=10, random_state=SEED, class_weight=cw); base.update(p)
        return DecisionTreeClassifier(**base)
    if name == "Logistic Regression":
        base = dict(C=1.0, max_iter=3000, random_state=SEED, class_weight=cw); base.update(p)
        return LogisticRegression(**base)
    if name == "KNN":
        base = dict(n_neighbors=15, weights="distance"); base.update(p)
        return KNeighborsClassifier(**base)
    if name == "Random Forest":
        base = dict(n_estimators=300, min_samples_leaf=2, n_jobs=1, random_state=SEED, class_weight=cw); base.update(p)
        return RandomForestClassifier(**base)
    if name == "XGBoost":
        base = dict(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                    tree_method="hist", n_jobs=1, random_state=SEED, eval_metric="mlogloss", objective="multi:softprob")
        base.update(p)
        return XGBClassifier(**base)
    raise ValueError(name)

def make_preprocessor(feats):
    nom = [c for c in NOMINAL if c in feats]
    rest = [c for c in feats if c not in nom]
    ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=15, sparse_output=False)
    return ColumnTransformer([("nom", ohe, nom), ("num", StandardScaler(), rest)], verbose_feature_names_out=False)

def build_pipeline(name, strategy, feats, params=None):
    steps = [("prep", make_preprocessor(feats))]
    if strategy == "smote":
        steps.append(("smote", SMOTE(random_state=SEED, k_neighbors=5)))
    steps.append(("model", make_estimator(name, strategy, params)))
    return Pipeline(steps)

def fit_params(name, strategy, y):
    """XGBoost + class_weight -> sample_weight seimbang (dihitung dari label latih saja)."""
    if name == "XGBoost" and strategy == "class_weight":
        return {"model__sample_weight": compute_sample_weight("balanced", y)}
    return {}

CLS_LAB = [0, 1, 2]
def full_metrics(y_true, y_pred, proba=None):
    d = {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"),
         "weighted_f1": f1_score(y_true, y_pred, average="weighted"), "balanced_acc": balanced_accuracy_score(y_true, y_pred),
         "mcc": matthews_corrcoef(y_true, y_pred)}
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=CLS_LAB, zero_division=0)
    for i, c in enumerate(CLASSES):
        d[f"prec_{c}"], d[f"rec_{c}"], d[f"f1_{c}"] = p[i], r[i], f[i]
    if proba is not None:
        d["auc_ovr"] = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    return d

SCORERS = {"accuracy": "accuracy", "macro_f1": "f1_macro", "weighted_f1": "f1_weighted", "bal_acc": "balanced_accuracy",
           "rec_drop":  make_scorer(recall_score,    labels=[0], average="macro", zero_division=0),
           "prec_drop": make_scorer(precision_score, labels=[0], average="macro", zero_division=0)}

# uji sanity pipeline: tidak ada kebocoran - SMOTE ada di dalam pipeline & preprocess di-fit per fold
_t = build_pipeline("XGBoost", "smote", FEATS_B)
print("Langkah pipeline contoh (XGBoost + SMOTE):", [s[0] for s in _t.steps])

### 5.2 Baseline: *Majority Class*
Pembanding minimum — selalu menebak kelas terbanyak. Model yang dibangun harus **jauh** melampaui ini pada Macro-F1 (bukan hanya akurasi).

In [ ]:
base_cv = cross_validate(DummyClassifier(strategy="most_frequent"), X_train[FEATS_A], y_train, cv=CV, scoring=SCORERS)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train[FEATS_A], y_train)
base_te = full_metrics(y_test, dummy.predict(X_test[FEATS_A]), dummy.predict_proba(X_test[FEATS_A]))
display(pd.DataFrame({"CV macro-F1": [base_cv["test_macro_f1"].mean()], "CV akurasi": [base_cv["test_accuracy"].mean()],
                      "Uji macro-F1": [base_te["macro_f1"]], "Uji akurasi": [base_te["accuracy"]], "Uji recall Dropout": [base_te["rec_Dropout"]]},
                     index=["Majority Class Baseline"]))

### 5.3 *Screening* menyeluruh: 3 skenario × 5 algoritma × 3 strategi imbalance
Seluruh kombinasi dievaluasi dengan **Stratified 5-Fold CV pada data latih** (hyperparameter default yang wajar; tuning dilakukan di Bab 6). Hasil ini sekaligus menjawab **RM-3** pada level validasi silang. Pipeline dijalankan paralel (satu fold = satu tugas).

* 3 skenario × 5 algoritma × 3 strategi = 45 konfigurasi, dikurangi 3 (KNN × class weight tidak tersedia) = **42**.
* Kolom `gap` = macro-F1 latih − validasi → indikator *overfitting*.

In [ ]:
def screen_one(sc, name, strat):
    feats = SCENARIOS[sc]
    pipe = build_pipeline(name, strat, feats)
    fp = fit_params(name, strat, y_train)
    r = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORERS, params=fp, return_train_score=True, n_jobs=1)
    out = dict(skenario=sc, model=name, strategi=strat, n_fitur=len(feats))
    for k in SCORERS:
        out[f"cv_{k}"] = r[f"test_{k}"].mean()
    out["cv_macro_f1_std"] = r["test_macro_f1"].std()
    out["train_macro_f1"] = r["train_macro_f1"].mean()
    out["gap"] = out["train_macro_f1"] - out["cv_macro_f1"]
    out["fit_sec"] = r["fit_time"].mean()
    return out

jobs = [(sc, m, s) for sc in SCENARIOS for m in MODEL_NAMES for s in STRATEGIES if not (m == "KNN" and s == "class_weight")]
t = time.time()
screen = pd.DataFrame(Parallel(n_jobs=CPU)(delayed(screen_one)(*j) for j in jobs))
print(f"{len(screen)} konfigurasi selesai dalam {time.time()-t:.0f} detik")
display(screen.sort_values(["skenario", "cv_macro_f1"], ascending=[True, False])
        [["skenario", "model", "strategi", "cv_macro_f1", "cv_macro_f1_std", "cv_accuracy", "cv_rec_drop", "cv_prec_drop", "gap"]].head(15))

In [ ]:
# Heatmap Macro-F1 (CV) dan Recall Dropout (CV): model x strategi, per skenario
def heat_screen(metric, title, cmap):
    fig, axs = plt.subplots(1, 3, figsize=(16, 3.9), sharey=True)
    vmin, vmax = screen[metric].min(), screen[metric].max()
    for ax, sc in zip(axs, SCENARIOS):
        pv = screen[screen.skenario == sc].pivot(index="model", columns="strategi", values=metric).reindex(index=MODEL_NAMES, columns=STRATEGIES)
        sns.heatmap(pv.rename(columns=STRAT_LABEL), annot=True, fmt=".3f", cmap=cmap, vmin=vmin, vmax=vmax, cbar=False, ax=ax, linewidths=.5,
                    annot_kws={"fontsize": 10})
        ax.set_title(SCEN_DESC[sc]); ax.set_ylabel(""); ax.set_xlabel("")
    fig.suptitle(title, y=1.04, fontweight="bold"); plt.tight_layout(); plt.show()

heat_screen("cv_macro_f1", "Macro-F1 (5-fold CV, data latih)", "YlGnBu")
heat_screen("cv_rec_drop", "Recall kelas Dropout (5-fold CV, data latih)", "OrRd")

# Strategi terbaik per (skenario, model) menurut Macro-F1 CV -> dipakai pada tahap tuning
idx = screen.groupby(["skenario", "model"])["cv_macro_f1"].idxmax()
best_strat = {(r.skenario, r.model): r.strategi for r in screen.loc[idx].itertuples()}
tbl = screen.loc[idx].pivot(index="model", columns="skenario", values="strategi").reindex(MODEL_NAMES).rename(columns=SCEN_DESC)
display(Markdown("**Strategi imbalance terbaik per model & skenario (berdasarkan Macro-F1 CV):**"))
display(tbl.replace(STRAT_LABEL))

In [ ]:
# Rata-rata efek strategi pada seluruh algoritma (ringkas RM-3 pada level CV)
eff = (screen.groupby(["skenario", "strategi"])[["cv_macro_f1", "cv_rec_drop", "cv_prec_drop", "cv_accuracy"]].mean()
       .rename(columns={"cv_macro_f1": "Macro-F1", "cv_rec_drop": "Recall Dropout", "cv_prec_drop": "Precision Dropout", "cv_accuracy": "Akurasi"}))
display(eff.round(4))

# Pemeriksaan overfitting
ov = screen.groupby("model")["gap"].mean().reindex(MODEL_NAMES).to_frame("rata-rata gap (latih - validasi)")
display(ov)

<a id="bab6"></a>
## 6. Penyetelan Hyperparameter (*Hyperparameter Tuning*)
[⬆ daftar isi](#toc)

**Protokol:** untuk setiap skenario dan setiap algoritma dipakai **strategi imbalance terbaik hasil screening** (Bab 5.3), lalu `RandomizedSearchCV` dengan **Stratified 5-Fold** dan metrik optimasi **Macro-F1**. `RandomizedSearchCV` dipilih karena pada ruang parameter lebar, pencarian acak lebih efisien daripada *grid* pada anggaran komputasi yang sama (Bergstra & Bengio, 2012). Pipeline utuh (termasuk preprocessing & SMOTE) dimasukkan ke pencarian sehingga **tidak ada kebocoran** antar fold.

*Pemilihan model akhir ("juara") tiap skenario dilakukan dari skor CV ini — bukan dari data uji.*

In [ ]:
SPACES = {
    "Decision Tree": dict(max_depth=randint(3, 16), min_samples_leaf=randint(2, 40), min_samples_split=randint(2, 40), criterion=["gini", "entropy"]),
    "Logistic Regression": dict(C=loguniform(1e-3, 1e2)),
    "KNN": dict(n_neighbors=randint(5, 60), weights=["uniform", "distance"], p=[1, 2]),
    "Random Forest": dict(n_estimators=[200, 300, 400], max_depth=[None, 8, 12, 16, 24], min_samples_leaf=randint(1, 10),
                          min_samples_split=randint(2, 15), max_features=["sqrt", "log2", 0.3, 0.5]),
    "XGBoost": dict(n_estimators=randint(150, 450), learning_rate=loguniform(0.01, 0.2), max_depth=randint(2, 8),
                    min_child_weight=randint(1, 10), subsample=uniform(0.6, 0.4), colsample_bytree=uniform(0.5, 0.5),
                    reg_lambda=loguniform(0.5, 20), gamma=uniform(0, 2)),
}
KEY = {"Decision Tree": "DT", "Logistic Regression": "LR", "KNN": "KNN", "Random Forest": "RF", "XGBoost": "XGB"}

tuned = {}
t_all = time.time()
for sc in SCENARIOS:
    for name in MODEL_NAMES:
        strat = best_strat[(sc, name)]
        pipe = build_pipeline(name, strat, SCENARIOS[sc])
        search = RandomizedSearchCV(pipe, {f"model__{k}": v for k, v in SPACES[name].items()}, n_iter=N_ITER[KEY[name]],
                                    scoring="f1_macro", cv=CV, random_state=SEED, n_jobs=CPU, refit=True)
        t = time.time()
        search.fit(X_train, y_train, **fit_params(name, strat, y_train))
        bi = search.best_index_
        tuned[(sc, name)] = dict(search=search, pipe=search.best_estimator_, strategy=strat, params=search.best_params_,
                                 cv_best=search.best_score_, cv_std=search.cv_results_["std_test_score"][bi])
        print(f"[{sc}] {name:<20s} strategi={STRAT_LABEL[strat]:<16s} CV macro-F1={search.best_score_:.4f}  ({time.time()-t:.0f} dtk)")
print(f"\nTuning total: {(time.time()-t_all)/60:.1f} menit")

In [ ]:
# Hyperparameter terbaik & skor CV (setelah tuning) vs skor default (screening)
rows = []
for (sc, name), v in tuned.items():
    d0 = screen[(screen.skenario == sc) & (screen.model == name) & (screen.strategi == v["strategy"])]["cv_macro_f1"].iloc[0]
    rows.append(dict(skenario=sc, model=name, strategi=STRAT_LABEL[v["strategy"]], cv_default=d0, cv_tuned=v["cv_best"], cv_std=v["cv_std"],
                     params={k.replace("model__", ""): (round(x, 4) if isinstance(x, float) else x) for k, x in v["params"].items()}))
tune_df = pd.DataFrame(rows)
tune_df["selisih"] = tune_df["cv_tuned"] - tune_df["cv_default"]
with pd.option_context("display.max_colwidth", 140):
    display(tune_df[["skenario", "model", "strategi", "cv_default", "cv_tuned", "selisih", "cv_std", "params"]])

# Juara tiap skenario (berdasarkan CV, BUKAN data uji)
champ = {sc: max(MODEL_NAMES, key=lambda m: tuned[(sc, m)]["cv_best"]) for sc in SCENARIOS}
display(Markdown("**Model juara per skenario (dipilih dari Macro-F1 CV):** " + " · ".join(
    f"**{SCEN_DESC[sc]}** → {champ[sc]} ({STRAT_LABEL[tuned[(sc, champ[sc])]['strategy']]}; CV = {tuned[(sc, champ[sc])]['cv_best']:.4f})" for sc in SCENARIOS)))

<a id="bab7"></a>
## 7. Evaluasi pada Data Uji (*Evaluation*)
[⬆ daftar isi](#toc)

Untuk pertama kalinya **data uji (20%)** dipakai. Setiap model hasil tuning dievaluasi **sekali**. Tabel utama memuat Macro-F1 (metrik utama) beserta metrik pendukung.

In [ ]:
PRED, PROBA, test_rows = {}, {}, []
for (sc, name), v in tuned.items():
    pipe = v["pipe"]
    PROBA[(sc, name)] = pipe.predict_proba(X_test)
    PRED[(sc, name)] = PROBA[(sc, name)].argmax(1)
    m = full_metrics(y_test, PRED[(sc, name)], PROBA[(sc, name)])
    m.update(skenario=sc, model=name, strategi=STRAT_LABEL[v["strategy"]], cv_macro_f1=v["cv_best"])
    test_rows.append(m)
for sc in SCENARIOS:        # baseline mayoritas
    m = dict(base_te); m.update(skenario=sc, model="Majority Baseline", strategi="-", cv_macro_f1=base_cv["test_macro_f1"].mean()); test_rows.append(m)
test_df = pd.DataFrame(test_rows)
ORDER = MODEL_NAMES + ["Majority Baseline"]

for sc in SCENARIOS:
    d = test_df[test_df.skenario == sc].set_index("model").reindex(ORDER)
    display(Markdown(f"#### Skenario {SCEN_DESC[sc]}"))
    display(d[["strategi", "cv_macro_f1", "macro_f1", "weighted_f1", "accuracy", "balanced_acc", "rec_Dropout", "prec_Dropout", "f1_Enrolled", "auc_ovr", "mcc"]]
            .rename(columns={"cv_macro_f1": "CV macro-F1", "macro_f1": "UJI macro-F1", "weighted_f1": "weighted-F1", "accuracy": "akurasi", "balanced_acc": "balanced acc",
                             "rec_Dropout": "recall Dropout", "prec_Dropout": "precision Dropout", "f1_Enrolled": "F1 Enrolled", "auc_ovr": "AUC OvR"})
            .style.format({c: "{:.4f}" for c in ["CV macro-F1", "UJI macro-F1", "weighted-F1", "akurasi", "balanced acc", "recall Dropout", "precision Dropout", "F1 Enrolled", "AUC OvR", "mcc"]}).background_gradient(subset=["UJI macro-F1"], cmap="Greens"))

In [ ]:
# Heatmap ringkas: Macro-F1 uji (model x skenario)
pv = test_df.pivot(index="model", columns="skenario", values="macro_f1").reindex(ORDER)
pa = test_df.pivot(index="model", columns="skenario", values="accuracy").reindex(ORDER)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
sns.heatmap(pv.rename(columns=SCEN_DESC), annot=True, fmt=".3f", cmap="YlGnBu", ax=axs[0], cbar=False, linewidths=.5); axs[0].set_title("Macro-F1 pada data uji"); axs[0].set_ylabel(""); axs[0].set_xlabel("")
sns.heatmap(pa.rename(columns=SCEN_DESC), annot=True, fmt=".3f", cmap="YlOrBr", ax=axs[1], cbar=False, linewidths=.5); axs[1].set_title("Akurasi pada data uji"); axs[1].set_ylabel(""); axs[1].set_xlabel("")
plt.tight_layout(); plt.show()

champ_test = {sc: test_df[(test_df.skenario == sc) & (test_df.model == champ[sc])].iloc[0] for sc in SCENARIOS}
display(pd.DataFrame({SCEN_DESC[sc]: {"juara (pilihan CV)": champ[sc], "UJI macro-F1": champ_test[sc]["macro_f1"], "UJI akurasi": champ_test[sc]["accuracy"],
                                      "recall Dropout": champ_test[sc]["rec_Dropout"], "precision Dropout": champ_test[sc]["prec_Dropout"],
                                      "AUC OvR": champ_test[sc]["auc_ovr"]} for sc in SCENARIOS}))

### 7.1 Confusion matrix model juara tiap skenario
Baris dinormalisasi per kelas sebenarnya (= *recall*); angka kecil di bawahnya adalah jumlah mahasiswa. Fokus: **ke mana *Enrolled* "tersasar"**.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.9))
for ax, sc in zip(axs, SCENARIOS):
    cm = confusion_matrix(y_test, PRED[(sc, champ[sc])], labels=CLS_LAB); cmn = cm / cm.sum(1, keepdims=True)
    sns.heatmap(cmn, annot=False, cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, linewidths=1, linecolor="white")
    for i in range(3):
        for j in range(3):
            ax.text(j + .5, i + .5, f"{cmn[i, j]*100:.1f}%\n(n={cm[i, j]})", ha="center", va="center", fontsize=10,
                    color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES, rotation=0); ax.set_xlabel("Prediksi"); ax.set_ylabel("Sebenarnya")
    ax.set_title(f"{SCEN_DESC[sc]}\n{champ[sc]} | macro-F1 = {champ_test[sc]['macro_f1']:.3f}")
plt.tight_layout(); plt.show()

rep = {sc: pd.DataFrame(classification_report(y_test, PRED[(sc, champ[sc])], target_names=CLASSES, output_dict=True, zero_division=0)).T.round(4) for sc in SCENARIOS}
for sc in SCENARIOS:
    display(Markdown(f"**Classification report — {SCEN_DESC[sc]} ({champ[sc]})**")); display(rep[sc])

### 7.2 Kurva ROC (OvR) dan Precision–Recall untuk kelas *Dropout*
Pada data tidak seimbang, kurva **PR** lebih informatif daripada ROC (Saito & Rehmsmeier, 2015) karena memperlihatkan *precision* pada tiap level *recall* — langsung relevan bagi biaya intervensi.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
yb = (y_test == 0).astype(int)
for sc in SCENARIOS:
    p0 = PROBA[(sc, champ[sc])][:, 0]
    fpr, tpr, _ = roc_curve(yb, p0); axs[0].plot(fpr, tpr, color=SCEN_PAL[sc], lw=2.2, label=f"{SCEN_DESC[sc]} (AUC={roc_auc_score(yb, p0):.3f})")
    pr, rc, _ = precision_recall_curve(yb, p0); axs[1].plot(rc, pr, color=SCEN_PAL[sc], lw=2.2, label=f"{SCEN_DESC[sc]} (AP={average_precision_score(yb, p0):.3f})")
axs[0].plot([0, 1], [0, 1], "k--", lw=1); axs[0].set_xlabel("False Positive Rate"); axs[0].set_ylabel("True Positive Rate"); axs[0].set_title("ROC — Dropout vs sisanya"); axs[0].legend(loc="lower right")
axs[1].axhline(yb.mean(), color="k", ls="--", lw=1, label=f"acak (prevalensi={yb.mean():.2f})"); axs[1].set_xlabel("Recall"); axs[1].set_ylabel("Precision"); axs[1].set_title("Precision-Recall — Dropout"); axs[1].legend(loc="lower left")
# AUC per kelas (OvR) per skenario
auc_cls = pd.DataFrame({SCEN_DESC[sc]: [roc_auc_score((y_test == k).astype(int), PROBA[(sc, champ[sc])][:, k]) for k in CLS_LAB] for sc in SCENARIOS}, index=CLASSES)
auc_cls.T.plot(kind="bar", ax=axs[2], color=[PAL[c] for c in CLASSES], rot=0, width=.75, edgecolor="white")
axs[2].set_ylim(.5, 1.0); axs[2].set_title("AUC OvR per kelas"); axs[2].set_ylabel("AUC"); axs[2].legend(title="")
for cont in axs[2].containers: axs[2].bar_label(cont, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

### 7.3 Interval kepercayaan *bootstrap* dan nilai tambah antar skenario (RM-2)
Satu angka dari satu data uji (n ≈ 885) mengandung ketidakpastian. Kami melakukan **bootstrap** pada data uji (1000 resampel; *percentile CI* 95%). Selisih antar skenario dihitung **berpasangan** (resampel indeks yang sama), sehingga CI selisih mencerminkan perbedaan nyata antar skenario.

In [ ]:
rng = np.random.default_rng(SEED); n_te = len(y_test)
BOOT_IDX = rng.integers(0, n_te, size=(N_BOOT, n_te))
def boot_metric(pred, fn):
    return np.array([fn(y_test[i], pred[i]) for i in BOOT_IDX])
mf1 = lambda yt, yp: f1_score(yt, yp, average="macro")
rdrop = lambda yt, yp: recall_score(yt, yp, labels=[0], average="macro", zero_division=0)
acc_f = lambda yt, yp: accuracy_score(yt, yp)
BOOT = {sc: {"macro_f1": boot_metric(PRED[(sc, champ[sc])], mf1), "rec_drop": boot_metric(PRED[(sc, champ[sc])], rdrop),
             "acc": boot_metric(PRED[(sc, champ[sc])], acc_f)} for sc in SCENARIOS}
ci = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))
ci_tbl = pd.DataFrame({SCEN_DESC[sc]: {"Macro-F1 (titik)": champ_test[sc]["macro_f1"], "Macro-F1 CI 95%": "[{:.3f}, {:.3f}]".format(*ci(BOOT[sc]["macro_f1"])),
                                       "Recall Dropout (titik)": champ_test[sc]["rec_Dropout"], "Recall Dropout CI 95%": "[{:.3f}, {:.3f}]".format(*ci(BOOT[sc]["rec_drop"])),
                                       "Akurasi (titik)": champ_test[sc]["accuracy"], "Akurasi CI 95%": "[{:.3f}, {:.3f}]".format(*ci(BOOT[sc]["acc"]))} for sc in SCENARIOS})
display(ci_tbl)

# selisih berpasangan antar skenario
diff_rows = []
for a, b in [("A", "B"), ("B", "C"), ("A", "C")]:
    for met in ["macro_f1", "rec_drop"]:
        d = BOOT[b][met] - BOOT[a][met]
        diff_rows.append(dict(perbandingan=f"{a} → {b}", metrik={"macro_f1": "Macro-F1", "rec_drop": "Recall Dropout"}[met], selisih=d.mean(),
                              ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5), P_perbaikan=(d > 0).mean()))
diff_df = pd.DataFrame(diff_rows); display(diff_df)

fig, ax = plt.subplots(figsize=(7.5, 4))
for i, sc in enumerate(SCENARIOS):
    lo, hi = ci(BOOT[sc]["macro_f1"]); ax.errorbar(i, champ_test[sc]["macro_f1"], yerr=[[champ_test[sc]["macro_f1"] - lo], [hi - champ_test[sc]["macro_f1"]]], fmt="o", ms=9, capsize=6, color=SCEN_PAL[sc], lw=2)
    ax.text(i + .08, champ_test[sc]["macro_f1"], f"{champ_test[sc]['macro_f1']:.3f}", va="center")
ax.axhline(base_te["macro_f1"], color="gray", ls=":", label=f"baseline mayoritas ({base_te['macro_f1']:.3f})")
ax.set_xticks(range(3)); ax.set_xticklabels([SCEN_DESC[s] for s in SCENARIOS]); ax.set_ylabel("Macro-F1 (data uji)"); ax.set_title("Nilai tambah informasi akademik (± CI bootstrap 95%)"); ax.legend(); ax.set_xlim(-.5, 2.6)
plt.tight_layout(); plt.show()

In [ ]:
dAB = diff_df.query("perbandingan=='A → B' and metrik=='Macro-F1'").iloc[0]; dBC = diff_df.query("perbandingan=='B → C' and metrik=='Macro-F1'").iloc[0]; dAC = diff_df.query("perbandingan=='A → C' and metrik=='Macro-F1'").iloc[0]
display(Markdown(f"""
**Jawaban RM-2 (otomatis dari data uji)**

* Skenario A (hanya data pendaftaran) mencapai Macro-F1 **{champ_test['A']['macro_f1']:.3f}** (CI {ci(BOOT['A']['macro_f1'])[0]:.3f}–{ci(BOOT['A']['macro_f1'])[1]:.3f}).
* Menambah semester 1 (B): **{champ_test['B']['macro_f1']:.3f}** → selisih **{dAB.selisih:+.3f}** (CI {dAB.ci_low:+.3f} s.d. {dAB.ci_high:+.3f}).
* Menambah semester 2 (C): **{champ_test['C']['macro_f1']:.3f}** → selisih B→C **{dBC.selisih:+.3f}** (CI {dBC.ci_low:+.3f} s.d. {dBC.ci_high:+.3f}); total A→C **{dAC.selisih:+.3f}** (CI {dAC.ci_low:+.3f} s.d. {dAC.ci_high:+.3f}).
* Recall Dropout: A **{champ_test['A']['rec_Dropout']:.3f}** → B **{champ_test['B']['rec_Dropout']:.3f}** → C **{champ_test['C']['rec_Dropout']:.3f}**.
* Seluruh skenario berada **jauh di atas baseline mayoritas** (Macro-F1 {base_te['macro_f1']:.3f}).
"""))

<a id="bab8"></a>
## 8. Pengaruh Penanganan Data Tidak Seimbang (RM-3)
[⬆ daftar isi](#toc)

Pertanyaan: apakah *class weight* dan SMOTE meningkatkan kemampuan mendeteksi **Dropout** dan **Enrolled** (minoritas), dan apa harganya pada *precision*?

Desain: untuk **Random Forest** dan **XGBoost** (dua algoritma utama proposal), hyperparameter hasil tuning dipakai tetap, lalu hanya **strategi** yang diganti (*none* / *class weight* / *SMOTE*), dilatih ulang pada data latih, dan dievaluasi pada data uji. Bukti pelengkap dari validasi silang tersedia pada Bab 5.3.

In [ ]:
imb_rows = []
for sc in SCENARIOS:
    for name in ["Random Forest", "XGBoost"]:
        prm = tuned[(sc, name)]["params"]
        for strat in STRATEGIES:
            pipe = build_pipeline(name, strat, SCENARIOS[sc], prm)
            pipe.fit(X_train, y_train, **fit_params(name, strat, y_train))
            pr = pipe.predict_proba(X_test); m = full_metrics(y_test, pr.argmax(1), pr)
            m.update(skenario=sc, model=name, strategi=strat); imb_rows.append(m)
imb = pd.DataFrame(imb_rows)
display(imb[["skenario", "model", "strategi", "macro_f1", "accuracy", "rec_Dropout", "prec_Dropout", "f1_Dropout", "rec_Enrolled", "prec_Enrolled", "f1_Enrolled", "rec_Graduate"]]
        .assign(strategi=lambda d: d.strategi.map(STRAT_LABEL)).round(4).set_index(["skenario", "model", "strategi"]))

fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (met, ttl) in zip(axs, [("rec_Dropout", "Recall Dropout"), ("prec_Dropout", "Precision Dropout"), ("macro_f1", "Macro-F1")]):
    g = imb.assign(lab=lambda d: d.skenario + "·" + d.model.map({"Random Forest": "RF", "XGBoost": "XGB"}))
    sns.barplot(data=g, x="lab", y=met, hue=g.strategi.map(STRAT_LABEL), hue_order=[STRAT_LABEL[s] for s in STRATEGIES], palette=["#95A5A6", "#E67E22", "#2E86C1"], ax=ax, edgecolor="white")
    lo = g[met].min(); ax.set_ylim(max(0, lo - .08), min(1, g[met].max() + .05)); ax.set_title(ttl); ax.set_xlabel("skenario·algoritma"); ax.set_ylabel("")
    if met != "rec_Dropout": ax.legend_.remove()
plt.tight_layout(); plt.show()

In [ ]:
MET = [("rec_Dropout", "recall Dropout"), ("prec_Dropout", "precision Dropout"), ("rec_Enrolled", "recall Enrolled"), ("prec_Enrolled", "precision Enrolled"),
       ("f1_Dropout", "F1 Dropout"), ("f1_Enrolled", "F1 Enrolled"), ("macro_f1", "macro-F1"), ("accuracy", "akurasi")]
delta = imb.pivot_table(index=["skenario", "model"], columns="strategi", values=[m for m, _ in MET])
rows = []
for strat in ["class_weight", "smote"]:
    rows.append(dict(strategi=STRAT_LABEL[strat], **{f"Δ {lab}": (delta[met][strat] - delta[met]["none"]).mean() for met, lab in MET}))
dtbl = pd.DataFrame(rows).set_index("strategi")
display(Markdown("**Rata-rata perubahan terhadap 'Tanpa penanganan' (6 kombinasi skenario × algoritma, data uji):**")); display(dtbl.round(4))
b_cw, b_sm = dtbl.loc["Class weight"], dtbl.loc["SMOTE"]
wins = {s: int(((delta['macro_f1'][s] - delta['macro_f1']['none']) > 0).sum()) for s in ["class_weight", "smote"]}

# pola 'recall naik, precision turun' untuk tiap kelas minoritas & strategi
pola = [f"{k} ({STRAT_LABEL[s]})" for k in ["Dropout", "Enrolled"] for s, b in [("class_weight", b_cw), ("smote", b_sm)]
        if b[f"Δ recall {k}"] > 0 and b[f"Δ precision {k}"] < 0]
pola_txt = ("terbukti pada: " + ", ".join(pola)) if pola else "tidak ditemukan pada rata-rata data ini"
display(Markdown(f"""
**Jawaban RM-3 (otomatis)** — perubahan rata-rata terhadap *tanpa penanganan*:

| | Δ recall Dropout | Δ precision Dropout | Δ recall Enrolled | Δ precision Enrolled | Δ macro-F1 | Δ akurasi | Macro-F1 membaik |
|---|---|---|---|---|---|---|---|
| Class weight | {b_cw['Δ recall Dropout']:+.3f} | {b_cw['Δ precision Dropout']:+.3f} | {b_cw['Δ recall Enrolled']:+.3f} | {b_cw['Δ precision Enrolled']:+.3f} | {b_cw['Δ macro-F1']:+.3f} | {b_cw['Δ akurasi']:+.3f} | {wins['class_weight']}/6 |
| SMOTE | {b_sm['Δ recall Dropout']:+.3f} | {b_sm['Δ precision Dropout']:+.3f} | {b_sm['Δ recall Enrolled']:+.3f} | {b_sm['Δ precision Enrolled']:+.3f} | {b_sm['Δ macro-F1']:+.3f} | {b_sm['Δ akurasi']:+.3f} | {wins['smote']}/6 |

* Pola yang diduga proposal (*recall kelas minoritas naik, precision turun*): **{pola_txt}**.
* Kelas minoritas sebenarnya adalah **Enrolled** (±18%), sedangkan Dropout (±32%) berada di tengah; penanganan imbalance paling kuat memengaruhi kelas yang bobotnya dinaikkan paling besar. Bila recall Dropout tidak naik, hal itu terjadi karena sebagian prediksi berpindah ke kelas Enrolled (lihat kolom Enrolled), bukan karena model gagal belajar.
* Penurunan akurasi saat Macro-F1 naik adalah **trade-off yang diharapkan**: model lebih adil antar kelas tetapi lebih sering "menebak" kelas minoritas.
"""))

<a id="bab9"></a>
## 9. Uji Signifikansi Statistik
[⬆ daftar isi](#toc)

Selisih Macro-F1 yang kecil (mis. RF vs XGBoost) bisa hanyalah *noise*. Prosedur (Demšar, 2006):

1. Skor Macro-F1 dari **Repeated Stratified 5-Fold CV** (2× ulang → 10 lipatan) pada data latih, untuk setiap model hasil tuning, dengan **lipatan identik** antar model (berpasangan).
2. **Uji Friedman** (apakah ada perbedaan antar 5 model) + **peringkat rata-rata**.
3. **Wilcoxon signed-rank berpasangan** model juara vs lainnya, dengan **koreksi Holm**.
4. Wilcoxon antar skenario (A vs B vs C) pada model juara masing-masing.

> Catatan: skor antar-lipatan tidak sepenuhnya independen (lipatan berbagi data latih; Nadeau & Bengio, 2003) sehingga p-value cenderung agak optimistis. Kami memakainya sebagai **indikator pendukung**, dilengkapi ukuran efek (selisih rata-rata).

In [ ]:
RCV = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=N_REPEATS_CV, random_state=SEED)
def rep_scores(sc, name):
    v = tuned[(sc, name)]
    return cross_val_score(clone(v["pipe"]), X_train, y_train, cv=RCV, scoring="f1_macro", params=fit_params(name, v["strategy"], y_train), n_jobs=CPU)

t = time.time()
REP = {(sc, m): rep_scores(sc, m) for sc in SCENARIOS for m in MODEL_NAMES}
print(f"Selesai dalam {time.time()-t:.0f} detik; {len(next(iter(REP.values())))} lipatan per model")

def holm(pvals):
    p = np.array(pvals); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(o):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

fr_rows, wil_rows = [], []
for sc in SCENARIOS:
    mat = np.column_stack([REP[(sc, m)] for m in MODEL_NAMES])
    stat, p = stats.friedmanchisquare(*[mat[:, j] for j in range(mat.shape[1])])
    ranks = pd.DataFrame(mat, columns=MODEL_NAMES).rank(axis=1, ascending=False).mean()
    fr_rows.append(dict(skenario=sc, chi2=stat, p_value=p, **{f"rank {m}": ranks[m] for m in MODEL_NAMES}))
    best = champ[sc]; others = [m for m in MODEL_NAMES if m != best]
    pv_, dm_ = [], []
    for o in others:
        try: pv_.append(stats.wilcoxon(REP[(sc, best)], REP[(sc, o)]).pvalue)
        except ValueError: pv_.append(1.0)
        dm_.append(REP[(sc, best)].mean() - REP[(sc, o)].mean())
    for o, p0, pa, dm in zip(others, pv_, holm(pv_), dm_):
        wil_rows.append(dict(skenario=sc, juara=best, pembanding=o, selisih_rata2=dm, p_wilcoxon=p0, p_Holm=pa, signifikan=("ya" if pa < .05 else "tidak")))
fr_df, wil_df = pd.DataFrame(fr_rows), pd.DataFrame(wil_rows)
display(Markdown("**Uji Friedman antar 5 algoritma (peringkat 1 = terbaik):**")); display(fr_df.round(4))
display(Markdown("**Wilcoxon berpasangan: juara vs model lain (koreksi Holm):**")); display(wil_df.round(4))

In [ ]:
# Perbandingan antar skenario (model juara masing-masing) - pada lipatan CV yang sama
sc_rows = []
for a, b in [("A", "B"), ("B", "C"), ("A", "C")]:
    xa, xb = REP[(a, champ[a])], REP[(b, champ[b])]
    try: p = stats.wilcoxon(xb, xa, alternative="greater").pvalue
    except ValueError: p = 1.0
    sc_rows.append(dict(perbandingan=f"{b} > {a}", selisih_macroF1_CV=xb.mean() - xa.mean(), p_wilcoxon_satu_sisi=p, lipatan_membaik=f"{int((xb > xa).sum())}/{len(xa)}"))
sc_df = pd.DataFrame(sc_rows); display(sc_df.round(5))

fig, ax = plt.subplots(figsize=(10, 4.4))
data = [(SCEN_DESC[sc], m, v) for sc in SCENARIOS for m in MODEL_NAMES for v in REP[(sc, m)]]
dd = pd.DataFrame(data, columns=["skenario", "model", "macro_f1"])
sns.boxplot(data=dd, x="skenario", y="macro_f1", hue="model", ax=ax, palette="Set2", fliersize=2, linewidth=1)
ax.set_title("Sebaran Macro-F1 antar-lipatan (Repeated Stratified 5-Fold)"); ax.set_xlabel(""); ax.set_ylabel("Macro-F1 (CV)"); ax.legend(ncol=5, loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

fr_best = {sc: fr_df.set_index("skenario").loc[sc] for sc in SCENARIOS}
rf_xgb = {sc: wil_df.query("skenario==@sc")  for sc in SCENARIOS}
display(Markdown("**Kesimpulan statistik (otomatis):** " + "; ".join(
    f"pada skenario {sc}, uji Friedman {'signifikan' if fr_best[sc]['p_value'] < .05 else 'tidak signifikan'} (p = {fr_best[sc]['p_value']:.2e}), juara = {champ[sc]}, "
    f"model lain yang tidak berbeda nyata dari juara: " + (", ".join(rf_xgb[sc].query("signifikan=='tidak'").pembanding) or "tidak ada") for sc in SCENARIOS) + "."))

<a id="bab10"></a>
## 10. ⭐ Analisis Lanjutan (di Atas Ekspektasi Proposal)
[⬆ daftar isi](#toc)

Bab ini menjawab pertanyaan yang biasanya diajukan penguji **setelah** angka akurasi ditampilkan: *"Kalau dipakai nyata, bagaimana? Bisa dipercaya? Adil? Apakah ada kebocoran? Bagaimana dibanding jurnal?"*

### 10.1 *Early-warning*: kurva gain/lift dan tabel threshold
Kampus tidak punya sumber daya membimbing semua mahasiswa. **Jika hanya k% mahasiswa dengan skor risiko tertinggi yang ditangani, berapa % dropout sungguhan yang tertangkap?** (*Capture rate*; lift = capture ÷ k.)

In [ ]:
def gains_curve(y_true, p_drop):
    order = np.argsort(-p_drop, kind="stable"); yb = (y_true[order] == 0).astype(int)
    return np.arange(1, len(yb) + 1) / len(yb), np.cumsum(yb) / yb.sum()

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
for sc in SCENARIOS:
    x, g = gains_curve(y_test, PROBA[(sc, champ[sc])][:, 0]); axs[0].plot(x * 100, g * 100, color=SCEN_PAL[sc], lw=2.3, label=SCEN_DESC[sc])
prev = (y_test == 0).mean(); axs[0].plot([0, 100], [0, 100], "k--", lw=1, label="acak")
axs[0].plot([0, prev * 100, 100], [0, 100, 100], color="gray", ls=":", lw=1.2, label="ideal")
axs[0].set_xlabel("% mahasiswa dengan skor risiko tertinggi yang ditangani"); axs[0].set_ylabel("% dropout yang tertangkap"); axs[0].set_title("Kurva gain — kelas Dropout (data uji)"); axs[0].legend(); axs[0].set_xlim(0, 100); axs[0].set_ylim(0, 102)

gk = []
for sc in SCENARIOS:
    x, g = gains_curve(y_test, PROBA[(sc, champ[sc])][:, 0])
    for k in (10, 20, 30, 40, 50):
        i = int(round(len(x) * k / 100)) - 1
        gk.append(dict(skenario=SCEN_DESC[sc], k=k, capture=g[i], lift=g[i] / (k / 100), precision_at_k=g[i] * prev / (k / 100)))
gk = pd.DataFrame(gk)
sns.barplot(data=gk, x="k", y="capture", hue="skenario", palette=[SCEN_PAL[s] for s in SCENARIOS], ax=axs[1], edgecolor="white")
axs[1].set_xlabel("% mahasiswa yang ditangani (k)"); axs[1].set_ylabel("proporsi dropout tertangkap"); axs[1].set_title("Capture rate pada k% teratas"); axs[1].set_ylim(0, 1.05)
for cont in axs[1].containers: axs[1].bar_label(cont, fmt="%.2f", fontsize=7)
plt.tight_layout(); plt.show()
display(gk.pivot(index="k", columns="skenario", values=["capture", "lift", "precision_at_k"]).round(3))

# tabel threshold probabilitas Dropout (deskriptif; memilih threshold sebaiknya dilakukan di data validasi, lihat Bab 12)
thr_rows = []
for sc in SCENARIOS:
    p0 = PROBA[(sc, champ[sc])][:, 0]
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        f = p0 >= t; tp = (f & (y_test == 0)).sum()
        thr_rows.append(dict(skenario=sc, threshold=t, ditandai_persen=f.mean() * 100, recall=tp / max(1, (y_test == 0).sum()), precision=tp / max(1, f.sum())))
display(pd.DataFrame(thr_rows).pivot(index="threshold", columns="skenario", values=["ditandai_persen", "recall", "precision"]).round(3))

g20 = gk.query("k==20").set_index("skenario")
display(Markdown("**Bacaan praktis (otomatis):** bila hanya **20%** mahasiswa berskor tertinggi yang diberi bimbingan, model menangkap **" + " / ".join(
    f"{g20.loc[SCEN_DESC[sc],'capture']*100:.0f}% ({SCEN_DESC[sc].split(':')[0]})" for sc in SCENARIOS) + "** dari seluruh dropout (acak hanya 20%).")) 

### 10.2 Kalibrasi probabilitas
Skor risiko dipakai sebagai angka ("peluang dropout 70%"), sehingga harus **terkalibrasi**. Perlu dicatat: *class weight* dan SMOTE memang menggeser probabilitas ke arah kelas minoritas, jadi kalibrasi sempurna tidak diharapkan; yang penting adalah **urutan risiko** (diagnosis gain/AUC) tetap baik dan pengguna mengetahui bias ini. Brier score lebih kecil = lebih baik.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.4))
brier = {}
for sc in SCENARIOS:
    p0 = PROBA[(sc, champ[sc])][:, 0]; yb = (y_test == 0).astype(int)
    fr, mp = calibration_curve(yb, p0, n_bins=8, strategy="quantile")
    ax.plot(mp, fr, "o-", color=SCEN_PAL[sc], lw=2, label=f"{SCEN_DESC[sc]} (Brier={brier_score_loss(yb, p0):.3f})")
    brier[sc] = brier_score_loss(yb, p0)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="kalibrasi sempurna")
ax.set_xlabel("Probabilitas Dropout prediksi (rata-rata per bin)"); ax.set_ylabel("Proporsi Dropout sebenarnya"); ax.set_title("Reliability diagram — Dropout"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
bs_ref = brier_score_loss((y_test == 0).astype(int), np.full(len(y_test), (y_train == 0).mean()))
display(pd.DataFrame({"Brier": brier, "Brier prediksi-konstan (prevalensi)": bs_ref, "Brier skill score": {sc: 1 - brier[sc] / bs_ref for sc in SCENARIOS}}).rename(index=SCEN_DESC).round(4))

### 10.3 Learning curve
Apakah model masih akan membaik bila datanya ditambah? Kurva yang sudah mendatar menandakan keterbatasan ada pada **informasi fitur**, bukan jumlah data.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2), sharey=True)
LC = {}
for ax, sc in zip(axs, SCENARIOS):
    v = tuned[(sc, champ[sc])]
    sizes, tr, va = learning_curve(clone(v["pipe"]), X_train, y_train, train_sizes=[.2, .4, .6, .8, 1.0], cv=CV, scoring="f1_macro",
                                   params=fit_params(champ[sc], v["strategy"], y_train), n_jobs=CPU, random_state=SEED)
    LC[sc] = (sizes, tr, va)
    ax.plot(sizes, tr.mean(1), "o-", color="#C0392B", label="latih"); ax.fill_between(sizes, tr.mean(1) - tr.std(1), tr.mean(1) + tr.std(1), color="#C0392B", alpha=.12)
    ax.plot(sizes, va.mean(1), "o-", color=SCEN_PAL[sc], label="validasi"); ax.fill_between(sizes, va.mean(1) - va.std(1), va.mean(1) + va.std(1), color=SCEN_PAL[sc], alpha=.15)
    ax.set_title(f"{SCEN_DESC[sc]} — {champ[sc]}"); ax.set_xlabel("jumlah data latih"); ax.legend(loc="lower right")
axs[0].set_ylabel("Macro-F1")
plt.tight_layout(); plt.show()
slope = {sc: (LC[sc][2].mean(1)[-1] - LC[sc][2].mean(1)[-2]) for sc in SCENARIOS}
display(Markdown("Kenaikan Macro-F1 validasi dari 80% → 100% data latih: " + ", ".join(f"**{sc}**: {slope[sc]:+.4f}" for sc in SCENARIOS) + ". Nilai yang mendekati nol berarti kurva sudah landai (penambahan data memberi manfaat kecil)."))

### 10.4 Ablasi grup fitur (skenario C)
Seberapa besar **setiap kelompok informasi** menyumbang kinerja? Dua eksperimen pada model juara skenario C (CV 5-fold, data latih):

* **Drop-one-group** — buang satu kelompok; makin besar penurunan, makin penting.
* **Only-one-group** — hanya satu kelompok; menunjukkan kekuatan prediksi mandiri.

Fitur rekayasa mengikuti kelompok sumbernya (fitur turunan sem-1 ikut dibuang bersama sem-1, dst.) supaya tidak ada informasi yang "bocor lewat pintu belakang".

In [ ]:
ENG_SRC = {"grade_gap": {"Jalur pendaftaran"},
           **{f: {"Akademik sem-1"} for f in ENG_B}, **{f: {"Akademik sem-2"} for f in ["sem2_approval_rate", "sem2_eval_pass_rate"]},
           **{f: {"Akademik sem-1", "Akademik sem-2"} for f in ["delta_grade", "delta_approved", "total_approved", "total_approval_rate"]}}
ABL_GROUPS = list(GROUPS.keys())

def feats_without(groups):
    gs = set(groups); keep = [c for c in FEATS_C if c in FEAT2GROUP and FEAT2GROUP[c] in GROUPS and FEAT2GROUP[c] not in gs and c not in ENG_SRC]
    keep += [c for c in FEATS_C if c in ENG_SRC and not (ENG_SRC[c] & gs)]
    return keep
def feats_only(group):
    keep = [c for c in GROUPS[group]]
    keep += [c for c, src in ENG_SRC.items() if c in FEATS_C and src <= {group}]
    return keep

cn = champ["C"]; vC = tuned[("C", cn)]
def abl_cv(feats):
    pipe = build_pipeline(cn, vC["strategy"], feats, vC["params"])
    return cross_val_score(pipe, X_train, y_train, cv=CV, scoring="f1_macro", params=fit_params(cn, vC["strategy"], y_train), n_jobs=CPU).mean()

full_f1 = abl_cv(FEATS_C)
abl = []
for g in ABL_GROUPS:
    abl.append(dict(kelompok=g, n_fitur_only=len(feats_only(g)), only=abl_cv(feats_only(g)), drop=abl_cv(feats_without([g]))))
abl = pd.DataFrame(abl).set_index("kelompok"); abl["Δ jika dibuang"] = abl["drop"] - full_f1
display(Markdown(f"Model: **{cn}** ({STRAT_LABEL[vC['strategy']]}); Macro-F1 CV seluruh fitur = **{full_f1:.4f}**"))
display(abl.sort_values("Δ jika dibuang")[["n_fitur_only", "only", "drop", "Δ jika dibuang"]].rename(columns={"only": "Macro-F1 hanya kelompok ini", "drop": "Macro-F1 tanpa kelompok ini"}))

fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
a = abl.sort_values("Δ jika dibuang", ascending=False)
axs[0].barh(a.index, a["Δ jika dibuang"], color=["#C0392B" if v < -0.005 else "#95A5A6" for v in a["Δ jika dibuang"]]); axs[0].set_title("Penurunan Macro-F1 bila kelompok dibuang"); axs[0].set_xlabel("Δ Macro-F1 (CV)")
b = abl.sort_values("only")
axs[1].barh(b.index, b["only"], color="#2E86C1"); axs[1].axvline(base_cv["test_macro_f1"].mean(), color="gray", ls=":", label="baseline mayoritas"); axs[1].axvline(full_f1, color="k", ls="--", label="semua fitur")
axs[1].set_title("Macro-F1 jika HANYA kelompok ini"); axs[1].set_xlabel("Macro-F1 (CV)"); axs[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

### 10.5 Sensitivitas *temporal leakage* pada skenario A
Skenario A diklaim sebagai **prediksi sebelum kuliah dimulai**. Namun beberapa atribut mungkin baru terekam **setelah** mahasiswa masuk:

* `Debtor` dan `Tuition fees up to date` — status pembayaran berkembang selama studi;
* `Unemployment rate`, `Inflation rate`, `GDP` — nilai makro pada periode pendaftaran bertindak sebagai **penanda angkatan** (*cohort fingerprint*), bukan sifat mahasiswa.

Dokumentasi dataset tidak menyatakan kapan tepatnya status pembayaran dicatat, sehingga kami **menguji sensitivitasnya**: jika kinerja anjlok saat dibuang, klaim "prediksi sebelum kuliah" perlu dibaca hati-hati.

In [ ]:
PAY = ["Debtor", "Tuition fees up to date"]; MACRO = GROUPS["Makroekonomi"]
variants = {"A (sesuai proposal)": FEATS_A, "A − status pembayaran": [c for c in FEATS_A if c not in PAY],
            "A − makroekonomi": [c for c in FEATS_A if c not in MACRO], "A − pembayaran − makro (A-ketat)": [c for c in FEATS_A if c not in PAY + MACRO]}
cA = champ["A"]; vA = tuned[("A", cA)]; rows = []
for lab, fs in variants.items():
    pipe = build_pipeline(cA, vA["strategy"], fs, vA["params"]); fp_ = fit_params(cA, vA["strategy"], y_train)
    cvs = cross_val_score(pipe, X_train, y_train, cv=CV, scoring="f1_macro", params=fp_, n_jobs=CPU).mean()
    pipe.fit(X_train, y_train, **fp_); pr = pipe.predict_proba(X_test); m = full_metrics(y_test, pr.argmax(1), pr)
    rows.append(dict(varian=lab, n_fitur=len(fs), cv_macro_f1=cvs, uji_macro_f1=m["macro_f1"], uji_recall_dropout=m["rec_Dropout"], uji_AUC=m["auc_ovr"]))
sens = pd.DataFrame(rows).set_index("varian"); sens["Δ macro-F1 uji vs A"] = sens["uji_macro_f1"] - sens.loc["A (sesuai proposal)", "uji_macro_f1"]
display(Markdown(f"Model skenario A: **{cA}** ({STRAT_LABEL[vA['strategy']]})")); display(sens)
strict = sens.loc["A − pembayaran − makro (A-ketat)"]
strict_msg = ("Penurunan relatif kecil: kemampuan prediksi dini bertumpu pada informasi yang memang tersedia saat pendaftaran." if abs(strict['Δ macro-F1 uji vs A']) < 0.02
              else "Penurunan cukup berarti: sebagian kinerja skenario A berasal dari informasi yang kemungkinan baru diketahui setelah masuk, sehingga klaim sebelum kuliah perlu dibaca lebih konservatif.")
display(Markdown(f"""
**Pembacaan (otomatis):** pada skenario A-ketat (tanpa pembayaran & makro) Macro-F1 uji = **{strict['uji_macro_f1']:.3f}** ({strict['Δ macro-F1 uji vs A']:+.3f} dari A).
{strict_msg}
Untuk sistem peringatan dini nyata, gunakan varian A-ketat sebagai acuan batas bawah yang jujur.
"""))

### 10.6 Perbandingan terukur dengan literatur
Angka 3-kelas dan biner tidak bisa dibandingkan langsung. Agar *apple-to-apple*, kami menjalankan dua tugas biner pada **pembagian data uji yang sama**:

* **Dropout vs Graduate** (seperti Mincevičius, 2026 — mahasiswa *Enrolled* dibuang);
* **Dropout vs sisanya** (seperti replikasi UCI pada Bettahi et al., 2025).

Model: XGBoost (hyperparameter hasil tuning skenario terkait, tanpa penanganan imbalance, objektif biner) dan Logistic Regression default. Kolom F1 dihitung untuk *kedua* kelas karena penulis rujukan tidak menyebut kelas positif untuk F1-nya pada ringkasan yang kami baca.

In [ ]:
def run_binary(task, sc, name):
    if task == "Dropout vs Graduate":
        tr, te = y_train != 1, y_test != 1
    else:
        tr, te = np.ones(len(y_train), bool), np.ones(len(y_test), bool)
    ytr, yte = (y_train[tr] == 0).astype(int), (y_test[te] == 0).astype(int)
    prm = {}
    if name == "XGBoost":
        prm = {k.replace("model__", ""): v for k, v in tuned[(sc, "XGBoost")]["params"].items()}
        prm.update(objective="binary:logistic", eval_metric="logloss")
    pipe = build_pipeline(name, "none", SCENARIOS[sc], prm); pipe.fit(X_train[tr], ytr)
    p = pipe.predict_proba(X_test[te])[:, 1]; pred = (p >= .5).astype(int)
    return dict(tugas=task, skenario=sc, model=name, akurasi=accuracy_score(yte, pred), AUC=roc_auc_score(yte, p),
                F1_Dropout=f1_score(yte, pred), F1_lawan=f1_score(yte, pred, pos_label=0), F1_weighted=f1_score(yte, pred, average="weighted"),
                recall_Dropout=recall_score(yte, pred))

bin_rows = [run_binary(t, sc, m) for t in ["Dropout vs Graduate", "Dropout vs sisanya"] for sc in SCENARIOS for m in ["XGBoost", "Logistic Regression"]]
bin_df = pd.DataFrame(bin_rows)
display(bin_df.set_index(["tugas", "skenario", "model"]).round(4))

lit = pd.DataFrame([
    dict(sumber="Mincevičius (2026) — LR, Dropout vs Graduate (DR_01)", tugas="Dropout vs Graduate", akurasi=0.905, AUC=0.96, F1="0,924 (kelas positif tidak dinyatakan pada ringkasan)"),
    dict(sumber="Mincevičius (2026) — RF + SHAP, Dropout vs Graduate", tugas="Dropout vs Graduate", akurasi=0.891, AUC=0.94, F1="0,914"),
    dict(sumber="Bettahi et al. (2025) — XGBoost, replikasi UCI (biner dropout)", tugas="Dropout vs sisanya", akurasi=np.nan, AUC=0.922, F1="0,792"),
    dict(sumber="Villar & de Andrade (2024) — CatBoost+Optuna, 3 kelas", tugas="3 kelas (F1 per kelas)", akurasi=np.nan, AUC="> 0,9", F1="0,86 / 0,88 / 0,84 (Graduate/Dropout/Enrolled)")])
display(Markdown("**Angka yang dilaporkan literatur (protokol masing-masing):**")); display(lit.set_index("sumber"))

c = bin_df.query("skenario=='C' and model=='XGBoost'").set_index("tugas")
display(Markdown(f"""
**Posisi hasil kami (skenario C, otomatis):**

* Dropout vs Graduate: akurasi **{c.loc['Dropout vs Graduate','akurasi']:.3f}**, AUC **{c.loc['Dropout vs Graduate','AUC']:.3f}** (rujukan LR: 0,905 / 0,96).
* Dropout vs sisanya: AUC **{c.loc['Dropout vs sisanya','AUC']:.3f}**, F1 Dropout **{c.loc['Dropout vs sisanya','F1_Dropout']:.3f}** (rujukan XGBoost: AUC 0,922 / F1 0,792).
* 3 kelas (juara C = {champ['C']}): F1 Dropout **{champ_test['C']['f1_Dropout']:.3f}**, F1 Graduate **{champ_test['C']['f1_Graduate']:.3f}**, F1 Enrolled **{champ_test['C']['f1_Enrolled']:.3f}**.

**Cara membaca:** pada tugas biner yang setara, selisih kami terhadap angka rujukan menunjukkan apakah hasil notebook ini berada dalam rentang yang wajar; selisih yang tersisa lazim berasal dari pembagian data, seleksi fitur, dan protokol resampling. Villar & de Andrade (2024) melaporkan F1 per kelas 0,83–0,88 pada 3 kelas; dari ringkasan yang kami baca kami tidak dapat memastikan kapan resampling diterapkan relatif terhadap pemisahan data, sehingga selisih terhadap hasil 3-kelas kami layak ditelusuri sebelum disimpulkan sebagai keunggulan metode. Pada notebook ini SMOTE **tidak pernah** menyentuh data validasi/uji.
"""))

### 10.7 Audit kewajaran subkelompok (*fairness audit*)
Sistem peringatan dini yang bias dapat merugikan kelompok tertentu (mis. gagal mendeteksi dropout pada satu gender, atau salah menandai terlalu banyak pada kelompok lain). Kami memeriksa **paritas tingkat deteksi** model juara skenario **B** (skenario peringatan dini paling realistis) pada data uji. Perbedaan **bukan bukti kausal**; subkelompok dengan n kecil ditandai sebagai tidak stabil.

In [ ]:
AUD_SC = "B"; aud_pred = PRED[(AUD_SC, champ[AUD_SC])]; Xa = X_test.copy().reset_index(drop=True)
Xa["Gender"] = Xa["Gender"].map({0: "Perempuan", 1: "Laki-laki"}); Xa["Scholarship holder"] = Xa["Scholarship holder"].map({0: "Tanpa beasiswa", 1: "Beasiswa"})
Xa["Daytime/evening attendance"] = Xa["Daytime/evening attendance"].map({1: "Kelas siang", 0: "Kelas malam"}); Xa["International"] = Xa["International"].map({0: "Domestik", 1: "Internasional"})
Xa["Debtor"] = Xa["Debtor"].map({0: "Bukan debitur", 1: "Debitur"}); Xa["Displaced"] = Xa["Displaced"].map({0: "Tidak perantau", 1: "Perantau"})
Xa["Kelompok usia"] = pd.cut(Xa["Age at enrollment"], [0, 19, 21, 25, 35, 100], labels=["<=19", "20-21", "22-25", "26-35", ">35"]).astype(str)
Xa["Program studi"] = Xa["Course"].map(COURSE_NAME)
rows = []
for col in ["Gender", "Kelompok usia", "Scholarship holder", "Daytime/evening attendance", "International", "Debtor", "Displaced", "Program studi"]:
    for g, idx_ in Xa.groupby(col).groups.items():
        idx_ = np.array(list(idx_)); yt, yp = y_test[idx_], aud_pred[idx_]
        nd = (yt == 0).sum()
        rows.append(dict(atribut=col, kelompok=g, n=len(idx_), n_dropout=nd, tingkat_dropout_nyata=(yt == 0).mean(), tingkat_ditandai=(yp == 0).mean(),
                         recall_dropout=((yp == 0) & (yt == 0)).sum() / nd if nd else np.nan, precision_dropout=((yp == 0) & (yt == 0)).sum() / max(1, (yp == 0).sum()),
                         akurasi=(yt == yp).mean(), stabil=("ya" if (len(idx_) >= 50 and nd >= 20) else "n kecil")))
aud = pd.DataFrame(rows)
overall_rec = recall_score(y_test, aud_pred, labels=[0], average="macro")
aud["selisih_recall_vs_total"] = aud["recall_dropout"] - overall_rec
display(Markdown(f"Skenario **{SCEN_DESC[AUD_SC]}**, model **{champ[AUD_SC]}**; recall Dropout keseluruhan = **{overall_rec:.3f}**"))
display(aud.set_index(["atribut", "kelompok"]).round(3))

stab = aud[aud.stabil == "ya"]
fig, ax = plt.subplots(figsize=(10, max(4, .28 * len(stab) + 1.5)))
s = stab.sort_values("recall_dropout")
ax.barh(s["atribut"] + " | " + s["kelompok"].astype(str), s["recall_dropout"], color=["#C0392B" if abs(v) > .10 else "#5D6D7E" for v in s["selisih_recall_vs_total"]])
ax.axvline(overall_rec, color="k", ls="--", label=f"keseluruhan ({overall_rec:.2f})"); ax.set_xlabel("Recall Dropout"); ax.set_title("Paritas deteksi Dropout per subkelompok (merah = selisih > 0,10)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
flag = stab[stab["selisih_recall_vs_total"].abs() > .10]
display(Markdown("**Subkelompok stabil dengan selisih recall > 0,10 dari rata-rata:** " + ("; ".join(f"{r.atribut} = {r.kelompok} (recall {r.recall_dropout:.2f}, n={r.n})" for r in flag.itertuples()) if len(flag) else "tidak ada.")))

In [ ]:
# Uji 'fairness through unawareness': apakah atribut sensitif dibutuhkan model?
SENS4 = ["Gender", "Nationality", "International", "Marital status"]; SENS5 = SENS4 + ["Age at enrollment"]
cB = champ["B"]; vB = tuned[("B", cB)]; rows = []
for lab, fs in [("B: seluruh fitur", FEATS_B), ("B tanpa gender/kebangsaan/internasional/status nikah", [c for c in FEATS_B if c not in SENS4]),
                ("B tanpa atribut di atas + usia", [c for c in FEATS_B if c not in SENS5])]:
    pipe = build_pipeline(cB, vB["strategy"], fs, vB["params"]); fp_ = fit_params(cB, vB["strategy"], y_train)
    pipe.fit(X_train, y_train, **fp_); pr = pipe.predict_proba(X_test); m = full_metrics(y_test, pr.argmax(1), pr)
    rows.append(dict(varian=lab, n_fitur=len(fs), uji_macro_f1=m["macro_f1"], uji_recall_dropout=m["rec_Dropout"], uji_AUC=m["auc_ovr"]))
ns = pd.DataFrame(rows).set_index("varian"); display(ns)
d_ns4 = ns.iloc[1, 1] - ns.iloc[0, 1]; d_ns5 = ns.iloc[2, 1] - ns.iloc[0, 1]
ns_msg = ("Membuang gender, kebangsaan, status internasional, dan status pernikahan hampir tidak menurunkan kinerja, sehingga atribut tersebut **tidak perlu** dimasukkan pada sistem nyata. "
          if abs(d_ns4) < 0.01 else "Membuang gender, kebangsaan, status internasional, dan status pernikahan menurunkan kinerja secara terukur; penggunaannya perlu pertimbangan etis dan kebijakan institusi. ")
ns_msg += ("Usia saat masuk membawa sinyal tambahan; gunakan sebagai konteks bagi dosen wali, bukan sebagai pemicu otomatis." if d_ns5 - d_ns4 < -0.01 else "Usia pun hampir tidak menambah kinerja.")
display(Markdown(f"Selisih Macro-F1 uji: tanpa 4 atribut sensitif = **{d_ns4:+.4f}**; tanpa 4 atribut + usia = **{d_ns5:+.4f}**. {ns_msg}"))

<a id="bab11"></a>
## 11. Interpretasi Model (RM-4)
[⬆ daftar isi](#toc)

Tiga pandangan yang saling melengkapi (sesuai proposal):

| Metode | Apa yang diukur | Kelemahan yang perlu diingat |
|---|---|---|
| **Feature Importance** (bawaan pohon) | Kontribusi fitur pada pembentukan pohon (*gain*/impurity) | Bias ke fitur kardinalitas tinggi; dihitung pada data latih |
| **Permutation Importance** | Penurunan Macro-F1 data uji saat satu fitur diacak | Fitur berkorelasi saling "menutupi" |
| **SHAP** | Kontribusi tiap fitur per prediksi (arah + besar), global & lokal | Mahal secara komputasi; bergantung model |

Karena fitur nominal di-*one-hot*, seluruh skor **diagregasi kembali ke fitur asli** agar bisa dibandingkan antar metode. Interpretasi memakai **model berbasis pohon terbaik** (RF/XGBoost) per skenario (menurut Macro-F1 CV) karena SHAP eksak tersedia untuk pohon.

In [ ]:
def orig_of(fname):
    for c in NOMINAL:
        if fname.startswith(c + "_"): return c
    return fname

def to3d(sv):
    """Seragamkan keluaran SHAP multikelas ke bentuk (n, fitur, kelas)."""
    if isinstance(sv, list): return np.stack(sv, axis=-1)
    sv = np.asarray(sv)
    if sv.ndim == 3 and sv.shape[-1] == 3: return sv
    if sv.ndim == 3 and sv.shape[0] == 3: return np.transpose(sv, (1, 2, 0))
    raise ValueError(sv.shape)

N_SHAP = 100 if FAST_MODE else 400
N_PERM = 3 if FAST_MODE else 10
INTERP = {}

def interpret(sc):
    tree_name = max(["Random Forest", "XGBoost"], key=lambda m: tuned[(sc, m)]["cv_best"])
    pipe = tuned[(sc, tree_name)]["pipe"]; feats = SCENARIOS[sc]
    prep, model = pipe.named_steps["prep"], pipe.named_steps["model"]
    Xt = pd.DataFrame(prep.transform(X_test[feats]), columns=prep.get_feature_names_out())
    orig = np.array([orig_of(n) for n in Xt.columns])
    # (1) feature importance bawaan
    fi = pd.Series(model.feature_importances_, index=Xt.columns).groupby(orig).sum()
    # (2) permutation importance (pada fitur asli, melalui pipeline penuh)
    pi = permutation_importance(pipe, X_test[feats], y_test, scoring="f1_macro", n_repeats=N_PERM, random_state=SEED, n_jobs=CPU)
    pim = pd.Series(pi.importances_mean, index=feats); pis = pd.Series(pi.importances_std, index=feats)
    # (3) SHAP
    rs = np.random.RandomState(SEED); ids = rs.choice(len(Xt), min(N_SHAP, len(Xt)), replace=False)
    expl = shap.TreeExplainer(model); sv = to3d(expl.shap_values(Xt.iloc[ids]))          # (n, f, 3)
    absm = np.abs(sv).mean(0)                                                                # (f, 3)
    shap_cls = pd.DataFrame(absm, index=Xt.columns, columns=CLASSES).groupby(orig).sum()
    shap_all = shap_cls.mean(axis=1)
    imp = pd.DataFrame({"FI_bawaan": fi, "Permutation": pim, "SHAP": shap_all}).fillna(0)
    return dict(sc=sc, tree=tree_name, pipe=pipe, Xt=Xt, ids=ids, sv=sv, expl=expl, orig=orig, imp=imp, pi_std=pis, shap_cls=shap_cls, feats=feats)

t = time.time()
for sc in ["A", "C"]:
    INTERP[sc] = interpret(sc); print(f"Skenario {sc}: interpretasi memakai {INTERP[sc]['tree']} ({time.time()-t:.0f} dtk)")

In [ ]:
def plot_importance(sc, topn=12):
    I = INTERP[sc]; imp = I["imp"]
    fig, axs = plt.subplots(1, 3, figsize=(18, 5.2))
    for ax, (col, ttl, colr) in zip(axs, [("FI_bawaan", "Feature Importance (bawaan model)", "#5D6D7E"), ("Permutation", "Permutation Importance\n(Δ Macro-F1 pada data uji)", "#2E86C1"), ("SHAP", "Mean |SHAP| (rata-rata 3 kelas)", "#1E8449")]):
        s = imp[col].sort_values(ascending=False).head(topn).iloc[::-1]
        err = I["pi_std"].reindex(s.index) if col == "Permutation" else None
        ax.barh(s.index, s.values, xerr=None if err is None else err.values, color=colr, capsize=2); ax.set_title(ttl)
    fig.suptitle(f"Faktor terpenting — {SCEN_DESC[sc]} ({I['tree']})", y=1.02, fontweight="bold"); plt.tight_layout(); plt.show()
plot_importance("C")
plot_importance("A")

In [ ]:
def triangulate(sc, k=10):
    imp = INTERP[sc]["imp"]; rk = imp.rank(ascending=False)
    sp = imp.corr(method="spearman")
    top = {c: set(imp[c].sort_values(ascending=False).head(k).index) for c in imp.columns}
    overlap = {f"{a} ∩ {b}": len(top[a] & top[b]) for a, b in itertools.combinations(imp.columns, 2)}
    cons = rk.mean(axis=1).sort_values(); cons_df = rk.loc[cons.index].assign(peringkat_rata2=cons.values).head(15).astype(float)
    return sp, overlap, cons_df

for sc in ["C", "A"]:
    sp, ov, cons_df = triangulate(sc)
    display(Markdown(f"#### Triangulasi 3 metode — {SCEN_DESC[sc]}"))
    display(Markdown(f"Korelasi peringkat Spearman antar metode:")); display(sp.round(3))
    display(Markdown("Irisan 10 fitur teratas antar metode: " + "; ".join(f"**{k}** = {v}/10" for k, v in ov.items())))
    display(Markdown("**15 fitur konsensus (peringkat rata-rata tiga metode; 1 = terpenting):**")); display(cons_df.round(1))

### 11.1 SHAP — arah dan besar pengaruh per kelas
Batang bertumpuk menunjukkan **kelas mana yang paling dipengaruhi** tiap fitur; *beeswarm* memperlihatkan **arah** (merah = nilai fitur tinggi) terhadap risiko *Dropout*.

In [ ]:
def plot_shap_class_stack(sc, topn=12):
    sc_df = INTERP[sc]["shap_cls"]; top = sc_df.sum(axis=1).sort_values(ascending=False).head(topn).index[::-1]
    fig, ax = plt.subplots(figsize=(9, 5.3)); left = np.zeros(len(top))
    for c in CLASSES:
        ax.barh(top, sc_df.loc[top, c], left=left, color=PAL[c], label=c, edgecolor="white"); left += sc_df.loc[top, c].values
    ax.set_xlabel("mean |SHAP|"); ax.set_title(f"Kontribusi fitur per kelas — {SCEN_DESC[sc]}"); ax.legend(); plt.tight_layout(); plt.show()
plot_shap_class_stack("C"); plot_shap_class_stack("A")

In [ ]:
def plot_beeswarm(sc, cls=0, max_display=15):
    I = INTERP[sc]
    try:
        plt.figure(figsize=(9, 6.2))
        shap.summary_plot(I["sv"][:, :, cls], I["Xt"].iloc[I["ids"]], max_display=max_display, show=False, plot_size=None)
        plt.title(f"SHAP beeswarm — kelas {CLASSES[cls]} — {SCEN_DESC[sc]} ({I['tree']})"); plt.tight_layout(); plt.show()
    except Exception as e:
        print("Beeswarm tidak dapat dirender:", e)
plot_beeswarm("C", 0); plot_beeswarm("C", 2); plot_beeswarm("A", 0)

In [ ]:
# Arah pengaruh pada fitur non-nominal: korelasi Spearman antara nilai fitur dan SHAP-nya (kelas Dropout)
def direction_table(sc, topn=10):
    I = INTERP[sc]; Xt, sv, ids = I["Xt"], I["sv"], I["ids"]; rows = []
    for f in I["imp"]["SHAP"].sort_values(ascending=False).index[:topn * 2]:
        if f in Xt.columns:                                 # non-nominal: satu kolom <-> satu fitur
            j = list(Xt.columns).index(f); xv = X_test.iloc[ids][f].values
            rho = stats.spearmanr(xv, sv[:, j, 0]).statistic
            rows.append(dict(fitur=f, rho_nilai_vs_SHAP_dropout=rho, arah=("nilai ↑ → risiko Dropout ↑" if rho > .1 else "nilai ↑ → risiko Dropout ↓" if rho < -.1 else "tidak monoton / lemah")))
    return pd.DataFrame(rows).head(topn).set_index("fitur")
for sc in ["C", "A"]:
    display(Markdown(f"**Arah pengaruh fitur numerik/biner teratas — {SCEN_DESC[sc]}:**")); display(direction_table(sc).round(3))

### 11.2 Kontribusi per **kelompok** informasi
Menjumlahkan |SHAP| per kelompok atribut menjawab pertanyaan kebijakan: *informasi apa yang paling menentukan, akademik atau latar belakang?*

In [ ]:
def group_share(sc):
    s = INTERP[sc]["imp"]["SHAP"]; g = s.groupby(lambda f: FEAT2GROUP.get(f, "Lainnya")).sum()
    return (g / g.sum() * 100).sort_values(ascending=False)
gs = {sc: group_share(sc) for sc in ["A", "C"]}
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, sc in zip(axs, ["A", "C"]):
    ax.barh(gs[sc].index[::-1], gs[sc].values[::-1], color=SCEN_PAL[sc])
    for i, v in enumerate(gs[sc].values[::-1]): ax.text(v + .5, i, f"{v:.1f}%", va="center")
    ax.set_title(f"Porsi kontribusi SHAP per kelompok — {SCEN_DESC[sc]}"); ax.set_xlabel("% dari total mean|SHAP|"); ax.set_xlim(0, max(gs[sc].values) * 1.18)
plt.tight_layout(); plt.show()
ACAD_FEATS = GROUPS["Akademik sem-1"] + GROUPS["Akademik sem-2"] + ENG_B + ENG_C
shC = INTERP["C"]["imp"]["SHAP"]; share_acad_C = shC[shC.index.isin(ACAD_FEATS)].sum() / shC.sum() * 100
print(f"Porsi data akademik (sem-1, sem-2, dan fitur rekayasa akademik) pada skenario C ≈ {share_acad_C:.1f}% dari total kontribusi SHAP.")

### 11.3 Penjelasan lokal: dua mahasiswa contoh
SHAP lokal menguraikan **satu prediksi**: fitur mana mendorong skor *Dropout* naik (merah) atau turun (biru) dari nilai dasar. Contoh 1: dropout yang **berhasil dideteksi**. Contoh 2: dropout yang **terlewat** — untuk memahami kegagalan model.

In [ ]:
def local_explain(sc, which="hit"):
    I = INTERP[sc]; pipe = tuned[(sc, I["tree"])]["pipe"]
    Xt_all = I["Xt"]; pr = pipe.predict_proba(X_test[I["feats"]]); pred = pr.argmax(1)
    cand = np.where((y_test == 0) & (pred == 0))[0] if which == "hit" else np.where((y_test == 0) & (pred != 0))[0]
    if len(cand) == 0: print("Tidak ada contoh."); return
    i = cand[np.argmax(pr[cand, 0])] if which == "hit" else cand[np.argmin(pr[cand, 0])]
    sv_i = to3d(I["expl"].shap_values(Xt_all.iloc[[i]]))[0, :, 0]
    base = float(np.atleast_1d(I["expl"].expected_value)[0])
    row = X_test.iloc[i]
    print(f"Skenario {sc} | indeks uji {i} | status sebenarnya: Dropout | P(Dropout)={pr[i,0]:.3f}, P(Enrolled)={pr[i,1]:.3f}, P(Graduate)={pr[i,2]:.3f} | prediksi: {CLASSES[pred[i]]}")
    try:
        ex = shap.Explanation(values=sv_i, base_values=base, data=Xt_all.iloc[i].values, feature_names=list(Xt_all.columns))
        shap.plots.waterfall(ex, max_display=12, show=False); plt.title(f"Penjelasan lokal SHAP (kelas Dropout) — {'terdeteksi' if which=='hit' else 'terlewat'}"); plt.tight_layout(); plt.show()
    except Exception as e:
        print("Waterfall tidak dapat dirender:", e)
local_explain("C", "hit"); local_explain("C", "miss")

In [ ]:
topC = INTERP["C"]["imp"].assign(rank=lambda d: d.rank(ascending=False).mean(axis=1)).sort_values("rank").head(6).index.tolist()
topA = INTERP["A"]["imp"].assign(rank=lambda d: d.rank(ascending=False).mean(axis=1)).sort_values("rank").head(6).index.tolist()
display(Markdown(f"""
**Jawaban RM-4 (otomatis, konsensus tiga metode):**

* **Skenario C (setelah semester 2)** — faktor teratas: {', '.join('*'+f+'*' for f in topC)}. Kelompok akademik menyumbang ≈ **{share_acad_C:.0f}%** total kontribusi SHAP.
* **Skenario A (saat pendaftaran)** — faktor teratas: {', '.join('*'+f+'*' for f in topA)}.
* Faktor **sosial-ekonomi & administratif** yang dapat dikendalikan kampus (mis. status uang kuliah, beasiswa) muncul sebagai pendukung, terutama pada skenario A dan B.
* Ketiga metode **sepakat** pada fitur-fitur teratas (lihat korelasi peringkat dan irisan top-10), sehingga kesimpulan tidak bergantung pada satu metode saja.
"""))

<a id="bab12"></a>
## 12. ⭐ Prototipe Sistem Peringatan Dini (*Risk Scoring*)
[⬆ daftar isi](#toc)

Model dikemas menjadi **skor risiko berjenjang** yang dapat langsung dipakai dosen wali:

* Ambang batas dipilih **hanya dari prediksi *out-of-fold* pada data latih** (bukan data uji) agar jujur:
  * **Tinggi**: `P(Dropout) ≥ t_tinggi`, yaitu ambang terkecil dengan *precision* OOF ≥ 0,70 (intervensi intensif; sedikit salah sasaran).
  * **Sedang**: `P(Dropout) ≥ t_sedang`, yaitu ambang terbesar dengan *recall* OOF ≥ 0,85 (pemantauan; jaring lebar).
  * **Rendah**: sisanya.
* Kemudian diuji apakah tingkat dropout **sebenarnya** pada data uji berurutan *Tinggi > Sedang > Rendah*.

In [ ]:
TIERS = {}
for sc in SCENARIOS:
    v = tuned[(sc, champ[sc])]
    oof = cross_val_predict(clone(v["pipe"]), X_train, y_train, cv=CV, method="predict_proba", params=fit_params(champ[sc], v["strategy"], y_train), n_jobs=CPU)[:, 0]
    yb = (y_train == 0).astype(int); ths = np.linspace(0.05, 0.95, 91)
    prec = np.array([yb[oof >= t].mean() if (oof >= t).sum() else np.nan for t in ths]); rec = np.array([((oof >= t) & (yb == 1)).sum() / yb.sum() for t in ths])
    ok_hi = ths[np.where(prec >= 0.70)[0]]; t_hi = ok_hi.min() if len(ok_hi) else 0.7
    ok_mid = ths[np.where(rec >= 0.85)[0]]; t_mid = min(ok_mid.max() if len(ok_mid) else 0.3, t_hi)
    TIERS[sc] = (t_mid, t_hi)

def tier_of(p, sc):
    t_mid, t_hi = TIERS[sc]; return np.where(p >= t_hi, "Tinggi", np.where(p >= t_mid, "Sedang", "Rendah"))

rows = []
for sc in SCENARIOS:
    p0 = PROBA[(sc, champ[sc])][:, 0]; tr = tier_of(p0, sc)
    for tname in ["Tinggi", "Sedang", "Rendah"]:
        m = tr == tname
        rows.append(dict(skenario=SCEN_DESC[sc], tingkat=tname, ambang=(f"≥ {TIERS[sc][1]:.2f}" if tname == "Tinggi" else f"≥ {TIERS[sc][0]:.2f}" if tname == "Sedang" else f"< {TIERS[sc][0]:.2f}"),
                         n=int(m.sum()), persen_mahasiswa=m.mean() * 100, dropout_nyata_persen=(y_test[m] == 0).mean() * 100 if m.sum() else np.nan,
                         persen_dari_semua_dropout=((y_test[m] == 0).sum() / (y_test == 0).sum()) * 100))
tier_df = pd.DataFrame(rows); display(tier_df.set_index(["skenario", "tingkat"]).round(1))

fig, ax = plt.subplots(figsize=(8, 4))
pv = tier_df.pivot(index="skenario", columns="tingkat", values="dropout_nyata_persen")[["Rendah", "Sedang", "Tinggi"]]
pv.plot(kind="bar", ax=ax, color=["#2471A3", "#E0A100", "#C0392B"], rot=0, width=.75, edgecolor="white")
for cont in ax.containers: ax.bar_label(cont, fmt="%.0f%%", fontsize=8)
ax.axhline(prev * 100, color="k", ls="--", lw=1, label=f"rata-rata dropout ({prev*100:.0f}%)"); ax.set_ylabel("% yang sebenarnya Dropout"); ax.set_title("Validasi tingkat risiko pada data uji"); ax.legend(loc="upper left", fontsize=8); ax.set_xlabel("")
plt.tight_layout(); plt.show()
mono = all(tier_df[tier_df.skenario == SCEN_DESC[sc]].set_index("tingkat").loc[["Rendah", "Sedang", "Tinggi"], "dropout_nyata_persen"].is_monotonic_increasing for sc in SCENARIOS)
print("Tingkat dropout nyata meningkat monoton Rendah < Sedang < Tinggi pada semua skenario:", mono)

In [ ]:
# Demonstrasi: kartu risiko untuk 6 mahasiswa uji acak (skenario B = peringatan dini setelah semester 1)
def risk_card(idx_list, sc="B"):
    pipe = tuned[(sc, champ[sc])]["pipe"]; Xs = X_test.iloc[idx_list]; pr = pipe.predict_proba(Xs)
    out = pd.DataFrame({"program studi": Xs["Course"].map(COURSE_NAME).values, "usia": Xs["Age at enrollment"].values,
                        "unit lulus sem-1": Xs["Curricular units 1st sem (approved)"].values, "uang kuliah lunas": Xs["Tuition fees up to date"].map({0: "tidak", 1: "ya"}).values,
                        "P(Dropout)": pr[:, 0], "P(Enrolled)": pr[:, 1], "P(Graduate)": pr[:, 2], "tingkat risiko": tier_of(pr[:, 0], sc),
                        "status sebenarnya": le.inverse_transform(y_test[idx_list])}, index=idx_list)
    return out
demo_idx = list(np.random.RandomState(7).choice(len(X_test), 6, replace=False))
display(Markdown(f"**Kartu risiko — {SCEN_DESC['B']} ({champ['B']}):**")); display(risk_card(demo_idx, "B").round(3))

<a id="bab13"></a>
## 13. Kesimpulan, Pengujian Hipotesis, Keterbatasan, dan Rekomendasi
[⬆ daftar isi](#toc)

### 13.1 Jawaban atas rumusan masalah (dihasilkan otomatis dari hasil di atas)

In [ ]:
# ----- RM-1 -----
avg_rank = fr_df.set_index("skenario")[[f"rank {m}" for m in MODEL_NAMES]].mean().rename(lambda s: s.replace("rank ", "")).sort_values()
rf_vs = {sc: tuned[(sc, "Random Forest")]["cv_best"] - tuned[(sc, "XGBoost")]["cv_best"] for sc in SCENARIOS}
rm1 = "\n".join(f"  * **{SCEN_DESC[sc]}**: juara menurut CV = **{champ[sc]}** (Macro-F1 uji {champ_test[sc]['macro_f1']:.3f}; akurasi {champ_test[sc]['accuracy']:.3f}). "
                f"Selisih CV RF − XGBoost = {rf_vs[sc]:+.4f}." for sc in SCENARIOS)
share_B = dAB.selisih / dAC.selisih * 100
sc_txt = ", ".join(f"{r.perbandingan}: p = {r.p_wilcoxon_satu_sisi:.3g}" for r in sc_df.itertuples())
rm1_close = ("Juara di ketiga skenario adalah model ensambel pohon (RF/XGBoost). " if all(champ[s] in ("Random Forest", "XGBoost") for s in SCENARIOS) else "Juara tidak konsisten berasal dari satu keluarga algoritma. ") + "Bila selisih dengan juara tidak signifikan, kesimpulan yang jujur adalah *setara secara statistik*, bukan *pasti lebih baik*."
ns_pairs = {sc: ", ".join(wil_df.query("skenario==@sc and signifikan=='tidak'").pembanding) or "tidak ada" for sc in SCENARIOS}
display(Markdown(f"""
**RM-1 — Algoritma terbaik.**
{rm1}
  * Peringkat rata-rata Friedman (semua skenario; makin kecil makin baik): {', '.join(f'{k} {v:.2f}' for k, v in avg_rank.items())}.
  * Model yang **tidak berbeda nyata** dari juara (Wilcoxon-Holm, α = 0,05): A → {ns_pairs['A']}; B → {ns_pairs['B']}; C → {ns_pairs['C']}.
  * Seluruh model jauh di atas *Majority Class Baseline* (Macro-F1 {base_te['macro_f1']:.3f}).
  * {rm1_close}

**RM-2 — Nilai tambah data akademik.** Macro-F1 uji meningkat dari **{champ_test['A']['macro_f1']:.3f} (A)** ke **{champ_test['B']['macro_f1']:.3f} (B)** dan **{champ_test['C']['macro_f1']:.3f} (C)**; kenaikan A→C = **{dAC.selisih:+.3f}** (CI 95% bootstrap {dAC.ci_low:+.3f} s.d. {dAC.ci_high:+.3f}). Recall Dropout naik dari **{champ_test['A']['rec_Dropout']:.3f}** ke **{champ_test['C']['rec_Dropout']:.3f}**. Uji Wilcoxon antar skenario: {sc_txt}.
Secara praktis, menunggu **satu semester** (B) sudah memberikan **{share_B:.0f}%** dari total kenaikan A→C; semester kedua (C) menambah sisanya.

**RM-3 — Penanganan data tidak seimbang** (rata-rata terhadap *tanpa penanganan*, data uji). *Class weight*: Δ recall Dropout {b_cw['Δ recall Dropout']:+.3f}, Δ precision Dropout {b_cw['Δ precision Dropout']:+.3f}, Δ recall Enrolled {b_cw['Δ recall Enrolled']:+.3f}, Δ Macro-F1 {b_cw['Δ macro-F1']:+.3f}. SMOTE: Δ recall Dropout {b_sm['Δ recall Dropout']:+.3f}, Δ precision Dropout {b_sm['Δ precision Dropout']:+.3f}, Δ recall Enrolled {b_sm['Δ recall Enrolled']:+.3f}, Δ Macro-F1 {b_sm['Δ macro-F1']:+.3f}. Pola recall-naik/precision-turun: {pola_txt}. Pilihan strategi sebaiknya mengikuti **biaya kesalahan** institusi: bila kehilangan satu dropout lebih mahal daripada satu bimbingan yang sia-sia, pilih strategi dengan recall Dropout lebih tinggi (lihat Bab 8 dan tabel threshold Bab 10.1).

**RM-4 — Faktor paling berpengaruh.** Skenario C: {', '.join('*'+f+'*' for f in topC)}; skenario A: {', '.join('*'+f+'*' for f in topA)}. Tiga metode interpretasi sepakat; data akademik semester awal mendominasi (≈ {share_acad_C:.0f}% kontribusi SHAP pada skenario C).
"""))

### 13.2 Pengujian hipotesis kerja proposal (Bab 6.3 proposal)
Hipotesis kerja diuji secara **objektif** terhadap hasil: status "didukung / tidak didukung" ditentukan oleh kriteria yang tertulis pada kolom *Kriteria*, bukan oleh opini.

In [ ]:
f1e = {sc: champ_test[sc][["f1_Dropout", "f1_Enrolled", "f1_Graduate"]] for sc in SCENARIOS}
h1 = (dAB.ci_low > 0) and (dAC.ci_low > 0)
h2 = share_acad_C > 50
h3 = all(f1e[sc].idxmin() == "f1_Enrolled" for sc in ["B", "C"])
h4 = len(pola) > 0
hyp = pd.DataFrame([
    ["H1", "Penambahan data sem-1 & sem-2 meningkatkan kinerja secara jelas dibanding A", "Batas bawah CI 95% selisih Macro-F1 (A→B dan A→C) > 0", f"A→B: {dAB.ci_low:+.3f}; A→C: {dAC.ci_low:+.3f}", "Didukung" if h1 else "Tidak didukung"],
    ["H2", "Fitur akademik mendominasi kepentingan fitur", "Porsi SHAP kelompok akademik (skenario C) > 50%", f"{share_acad_C:.1f}%", "Didukung" if h2 else "Tidak didukung"],
    ["H3", "Kelas Enrolled paling sulit diprediksi", "F1 Enrolled terendah di antara 3 kelas pada skenario B dan C", f"F1 (B): D={f1e['B']['f1_Dropout']:.2f}, E={f1e['B']['f1_Enrolled']:.2f}, G={f1e['B']['f1_Graduate']:.2f}; (C): D={f1e['C']['f1_Dropout']:.2f}, E={f1e['C']['f1_Enrolled']:.2f}, G={f1e['C']['f1_Graduate']:.2f}", "Didukung" if h3 else "Tidak didukung"],
    ["H4", "Penanganan imbalance menaikkan recall minoritas dengan kemungkinan turunnya precision", "Untuk setidaknya satu kelas minoritas (Dropout/Enrolled) dan satu strategi: rata-rata Δ recall > 0 dan Δ precision < 0", f"CW: Dropout Δrec {b_cw['Δ recall Dropout']:+.3f}/Δprec {b_cw['Δ precision Dropout']:+.3f}, Enrolled Δrec {b_cw['Δ recall Enrolled']:+.3f}/Δprec {b_cw['Δ precision Enrolled']:+.3f}; SMOTE: Dropout Δrec {b_sm['Δ recall Dropout']:+.3f}/Δprec {b_sm['Δ precision Dropout']:+.3f}, Enrolled Δrec {b_sm['Δ recall Enrolled']:+.3f}/Δprec {b_sm['Δ precision Enrolled']:+.3f}", "Didukung" if h4 else "Tidak didukung"]],
    columns=["Kode", "Hipotesis", "Kriteria", "Hasil", "Status"]).set_index("Kode")
with pd.option_context("display.max_colwidth", 200): display(hyp)

# analisis kesalahan: Enrolled "tersasar" ke mana?
cmC = confusion_matrix(y_test, PRED[("C", champ["C"])], labels=CLS_LAB); cmB = confusion_matrix(y_test, PRED[("B", champ["B"])], labels=CLS_LAB)
for nm, cm in [("B", cmB), ("C", cmC)]:
    r = cm[1] / cm[1].sum(); print(f"Skenario {nm}: dari mahasiswa Enrolled sebenarnya, {r[0]*100:.1f}% diprediksi Dropout, {r[1]*100:.1f}% Enrolled, {r[2]*100:.1f}% Graduate.")

### 13.3 Keterbatasan (disampaikan secara terbuka)

1. **Dataset dari Portugal (satu institusi), bukan Indonesia.** Dataset diposisikan sebagai *benchmark*; temuan metodologis (alur, evaluasi, skenario waktu) dapat dipindahkan, tetapi **angka dan peringkat faktor belum tentu berlaku** di kampus Indonesia.
2. **Kelas *Enrolled* secara konseptual ambigu** (mahasiswa yang masih studi di akhir periode normal; masa depannya belum jelas) sehingga batas kelasnya kabur — batas atas kinerja 3-kelas memang terbatas.
3. **Satu pembagian latih/uji (80/20).** Ketidakpastian dikuantifikasi dengan bootstrap & CV berulang, tetapi tidak ada validasi **temporal** (latih pada angkatan lama, uji pada angkatan baru) karena dataset tidak menyediakan tahun angkatan secara eksplisit; variabel makroekonomi bisa menjadi proxy angkatan.
4. **Waktu perekaman sebagian fitur tidak terdokumentasi** (mis. status pembayaran) → sensitivitas pada Bab 10.5.
5. **SMOTE pada fitur hasil one-hot** menghasilkan nilai pecahan pada kolom biner; SMOTE-NC/ADASYN/*cost-sensitive learning* dapat dicoba.
6. **Hyperparameter** dicari dengan anggaran terbatas (jumlah iterasi pada Bab 0) dan strategi imbalance dipilih sebelum tuning; pencarian lebih luas/ Optuna dapat menambah sedikit kinerja.
7. **Interpretasi SHAP bukan kausalitas.** Misalnya, status uang kuliah yang menunggak berasosiasi dengan dropout, tetapi tidak membuktikan bahwa melunasi akan mencegahnya.
8. **Kalibrasi** probabilitas terpengaruh class weight/SMOTE; sebelum dipakai nyata sebaiknya dikalibrasi (isotonic/Platt) pada data validasi lokal.

### 13.4 Rekomendasi

* **Untuk kampus:** gunakan **skenario B** (akhir semester 1) sebagai titik peringatan dini utama, dengan tiga tingkat risiko (Bab 12); skenario A cocok hanya untuk *screening* awal bimbingan orientasi, dengan catatan sensitivitas pada Bab 10.5.
* **Untuk kebijakan:** fokus intervensi pada **kemajuan akademik semester awal** (unit lulus & evaluasi) dan faktor administratif yang dapat dibantu (beasiswa, kemudahan pembayaran), bukan pada atribut demografis.
* **Untuk etika:** jangan memakai gender/kebangsaan/status pernikahan sebagai pemicu keputusan; biaya kinerja bila atribut tersebut dibuang diuji pada Bab 10.7. Gunakan skor sebagai **alat bantu dosen wali**, bukan keputusan otomatis.
* **Pengembangan lanjutan:** validasi temporal pada data kampus sendiri; SMOTE-NC/cost-sensitive; CatBoost/LightGBM; model survival untuk memprediksi *kapan* dropout terjadi; kalibrasi probabilitas; *concept drift monitoring*.

### 13.5 Penutup
Penelitian ini menunjukkan bahwa **prediksi dini dropout layak dilakukan** pada tiga tahap waktu, bahwa **informasi akademik satu semester** sudah menaikkan kemampuan deteksi secara berarti, dan bahwa evaluasi yang benar untuk data tidak seimbang (Macro-F1, recall per kelas, SMOTE tanpa kebocoran, interval kepercayaan, uji signifikansi) mengubah cara kita membaca "akurasi". Pipeline ini bersifat **dapat diulang, transparan, dan dapat dijelaskan** sehingga siap diadaptasi pada data kampus lain.

<a id="bab14"></a>
## 14. Daftar Pustaka
[⬆ daftar isi](#toc)

**Dataset & studi pada dataset yang sama**

1. Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). *Predict Students' Dropout and Academic Success* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5MC89
2. Realinho, V., Machado, J., Baptista, L., & Martins, M. V. (2022). Predicting student dropout and academic success. *Data*, 7(11), 146. MDPI. (Dataset juga di Zenodo: https://doi.org/10.5281/zenodo.5777339)
3. Martins, M. V., Tolledo, D., Machado, J., Baptista, L. M. T., & Realinho, V. (2021). Early prediction of student's performance in higher education: a case study. In *Trends and Applications in Information Systems and Technologies (WorldCIST 2021)*. Springer. https://doi.org/10.1007/978-3-030-72657-7_16
4. Villar, A., & de Andrade, C. R. V. (2024). Supervised machine learning algorithms for predicting student dropout and academic success: a comparative study. *Discover Artificial Intelligence*. https://doi.org/10.1007/s44163-023-00079-z
5. Bettahi, A., Belouadha, F.-Z., & Harroud, H. (2025). A modular and explainable machine learning pipeline for student dropout prediction in higher education. *Algorithms*, 18(10), 662. https://doi.org/10.3390/a18100662
6. Mincevičius, A. (2026). Interpretable machine learning approaches for student dropout prediction in higher education. *New Trends in Computer Sciences*, 4(1), 146–160. https://doi.org/10.3846/ntcs.2026.27184
7. Mustofa, S., Emon, Y. R., Mamun, S. B., Akhy, S. A., & Ahad, M. T. (2025). A novel AI-driven model for student dropout risk analysis with explainable AI insights. *Computers and Education: Artificial Intelligence*, 8, 100352. https://doi.org/10.1016/j.caeai.2024.100352

**Metode**

8. Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research*, 16, 321–357.
9. Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
10. Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proc. 22nd ACM SIGKDD*, 785–794. https://doi.org/10.1145/2939672.2939785
11. Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems (NeurIPS)*, 30.
12. Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research*, 13, 281–305.
13. Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research*, 7, 1–30.
14. Nadeau, C., & Bengio, Y. (2003). Inference for the generalization error. *Machine Learning*, 52(3), 239–281.
15. Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE*, 10(3), e0118432.
16. Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate. *Journal of the Royal Statistical Society B*, 57(1), 289–300.
17. Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proc. PAKDD*.
18. Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
19. Lemaître, G., Nogueira, F., & Aridas, C. K. (2017). Imbalanced-learn: A Python toolbox to tackle the curse of imbalanced datasets in machine learning. *Journal of Machine Learning Research*, 18(17), 1–5.

> Catatan: butir 1–7 diperiksa sumbernya pada penelusuran 20 Nov 2026 (butir 2 tidak menampilkan DOI pada halaman indeks yang dibuka; silakan cek ulang halaman MDPI saat finalisasi). Butir 8–19 adalah rujukan metodologis standar; periksa ulang volume/halaman saat memformat sesuai gaya sitasi dosen.

<a id="bab15"></a>
## 15. Lampiran: Reproduktibilitas
[⬆ daftar isi](#toc)

In [ ]:
total_min = (time.time() - T0) / 60
env = pd.Series({"Python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "imbalanced-learn": imblearn.__version__,
                 "shap": shap.__version__, "pandas": pd.__version__, "numpy": np.__version__, "SEED": SEED, "TEST_SIZE": TEST_SIZE, "CV": f"Stratified {CV_FOLDS}-Fold",
                 "N_ITER tuning": str(N_ITER), "N_BOOT": N_BOOT, "FAST_MODE": FAST_MODE, "Sumber data": SRC, "Waktu eksekusi total (menit)": round(total_min, 1)}, name="nilai")
display(env.to_frame())
print("Seluruh sel selesai dijalankan tanpa error.")